# Neural Diagnosis — GraphRAG
### A pipeline-first, neurology-specific medical diagnostic assistant

**Inspired by** *MED-COPILOT: A Medical Assistant Powered by GraphRAG and Similar Patient Case Retrieval* —
this is a **refined, neurology-focused extension**, not a reproduction.

**Differences from the base paper / our design choices:**
- No UI / frontend copilot — **notebook pipeline only**.
- **Neurology corpus** focus with evidence-type ranking.
- **Semantic chunking** that preserves section/paragraph boundaries + provenance.
- **Medical entity normalization** with a neurology lexicon (symptoms, diagnoses, drugs, imaging, red flags …).
- **Neurology knowledge-graph** construction with typed relations + per-edge provenance.
- **Hybrid retrieval** (dense FAISS + sparse BM25) fused into one ranked list.
- **Graph / path retrieval** returning provenance-aware subgraphs.
- **Cross-encoder + graph-aware reranker** combining dense, sparse, cross-encoder, graph-support and evidence-priority signals.
- **Provenance-preserving evidence aggregator** separating supporting / conflicting / missing evidence.
- **LLM with a strict structured diagnostic template** (provider-agnostic, with offline mock fallback).
- **Safety verifier** with neurology emergency rules (stroke, status epilepticus, meningitis, raised ICP, thunderclap headache).
- Optional, pluggable hook for *similar patient case retrieval* (left as a TODO).

```
Neurology Corpus
  -> cleaning + dedup + evidence ranking
  -> semantic chunking
  -> medical entity normalization
  -> neurology KG construction
  -> embeddings + FAISS
  -> hybrid retriever
  -> graph/path retriever
  -> cross-encoder + graph-aware reranker
  -> evidence aggregator
  -> LLM with structured diagnostic template
  -> safety verifier
  -> final answer with citations / confidence / escalation
```

> **How to run:** Execute cells top-to-bottom. The PRE-CELL downloads the notebook
> ONLY corpus — `OpenMed-Community/synthetic-neurology-conversations` from Hugging Face.
> **Internet must be ON.** There is no synthetic fallback; loading must succeed.


### Cell 0 — Install & Import Dependencies, Global Config

**Purpose:** Install the few extra libs Kaggle images don't ship by default, import everything,
and define global config + paths. All heavy biomedical models are *optional* with graceful fallbacks.

**Inputs:** none.
**Outputs:** `CONFIG` dict, `WORK_DIR`, `INPUT_DIR`, printed dependency report.
**Saved artifacts:** none.
**Customize later:** swap embedding model, cross-encoder, chunk size, top-k.


In [1]:
# ===== Neural Diagnosis - GraphRAG : Cell 0 - Setup =====
import os, sys, json, re, hashlib, math, warnings, pickle, textwrap, time
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")

# --- Optional installs (quiet). These are lightweight / pure-python where possible. ---
def _pip_install(pkgs):
    import subprocess
    for p in pkgs:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p],
                           check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            print(f"  [warn] could not install {p}: {e}")

# rank_bm25 = pure-python BM25. networkx ships on Kaggle. faiss-cpu + sentence-transformers
# are the two heavier deps; we try to install but tolerate failure (fallbacks exist).
# datasets/pyarrow: load HF conversational corpus (pre-cell).
# groq + google-generativeai: primary LLM providers (mock fallback if no key).
_pip_install(["rank_bm25", "faiss-cpu", "sentence-transformers",
              "datasets", "pyarrow", "groq", "google-generativeai"])

# --- Core scientific stack (all present on Kaggle) ---
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

# --- Optional heavy deps with fallback flags ---
HAVE_ST = False
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAVE_ST = True
except Exception:
    print("[info] sentence-transformers not available - will use TF-IDF/hash fallback embedder.")

HAVE_FAISS = False
try:
    import faiss
    HAVE_FAISS = True
except Exception:
    print("[info] faiss not available - will use numpy cosine fallback index.")

try:
    from rank_bm25 import BM25Okapi
    HAVE_BM25 = True
except Exception:
    print("[info] rank_bm25 not available - will use a simple TF sparse fallback.")
    HAVE_BM25 = False

# --- Kaggle path conventions ---
IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path("./input")
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR = WORK_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Retrieval / ranking
    "chunk_size": 320,          # target chars per chunk
    "chunk_overlap": 60,
    "top_k_hybrid": 12,
    "top_k_rerank": 6,
    # Models (override with API / HF models later)
    "embed_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "crossencoder_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "embed_dim": 384,
    # Hybrid fusion weights (dense vs sparse) before reranking
    "dense_weight": 0.6,
    "sparse_weight": 0.4,
    # Reranker weights
    "w_dense": 0.25, "w_sparse": 0.15, "w_cross": 0.35,
    "w_graph": 0.15, "w_priority": 0.10,
    # LLM (primary providers: Groq / Gemini; deterministic mock fallback)
    "llm_provider": "mock",     # auto-set below from LLM_PROVIDER
    "llm_model": "llama-3.3-70b-versatile",  # Groq default; gemini uses "gemini-1.5-flash"
    "max_tokens": 700,
}

# --- On Kaggle, read secrets via UserSecretsClient (falls back to env vars locally). ---
def _load_secret(name, default=""):
    """Read a secret from Kaggle Secrets if present, else from environment."""
    # environment first (works locally / CI)
    val = os.environ.get(name, "")
    if val:
        return val
    # Kaggle Secrets
    try:
        from kagglesecrets import UserSecretsClient
        sc = UserSecretsClient()
        try:
            return sc.get_secret(name) or ""
        except Exception:
            return ""
    except Exception:
        return default

# Configurable LLM provider. Set LLM_PROVIDER env / Kaggle Secret to "groq" or "gemini".
# Falls back to deterministic MOCK generation when the matching key is absent.
LLM_PROVIDER = (_load_secret("LLM_PROVIDER") or os.environ.get("LLM_PROVIDER", "groq")).lower()

# API keys: Kaggle Secrets (UserSecretsClient) -> env vars. Do NOT hardcode secrets.
# NOTE: OpenAI/Anthropic are intentionally NOT default dependencies; use Gemini/Groq.
API_KEYS = {
    "GEMINI_API_KEY":   _load_secret("GEMINI_API_KEY"),
    "GROQ_API_KEY":     _load_secret("GROQ_API_KEY"),
    "HF_TOKEN":         _load_secret("HF_TOKEN"),
    "PINECONE_API_KEY": _load_secret("PINECONE_API_KEY"),
}

# Resolve provider: only honor groq/gemini if their key is actually present.
if LLM_PROVIDER == "groq" and not API_KEYS["GROQ_API_KEY"]:
    LLM_PROVIDER = "mock"
elif LLM_PROVIDER == "gemini" and not API_KEYS["GEMINI_API_KEY"]:
    LLM_PROVIDER = "mock"
CONFIG["llm_provider"] = LLM_PROVIDER if LLM_PROVIDER in ("groq", "gemini") else "mock"

print("=" * 64)
print("Neural Diagnosis - GraphRAG  ::  environment report")
print("=" * 64)
print(f"Kaggle runtime        : {IS_KAGGLE}")
print(f"INPUT_DIR             : {INPUT_DIR}")
print(f"WORK_DIR              : {WORK_DIR}")
print(f"sentence-transformers : {HAVE_ST}")
print(f"faiss                 : {HAVE_FAISS}")
print(f"rank_bm25             : {HAVE_BM25}")
print(f"networkx version      : {nx.__version__}")
print(f"API keys configured   : {[k for k,v in API_KEYS.items() if v] or 'NONE (using mock LLM + local embeddings)'}")
print(f"LLM provider (req)    : {LLM_PROVIDER}")
print(f"LLM provider (active) : {CONFIG['llm_provider']}")
print("Config OK.")


Neural Diagnosis - GraphRAG  ::  environment report
Kaggle runtime        : True
INPUT_DIR             : /kaggle/input
WORK_DIR              : /kaggle/working
sentence-transformers : True
faiss                 : True
rank_bm25             : True
networkx version      : 3.6.1
API keys configured   : NONE (using mock LLM + local embeddings)
LLM provider (req)    : mock
LLM provider (active) : mock
Config OK.


### PRE-CELL — Load Hugging Face Neurology Conversations Dataset

**Purpose:** Download the primary corpus `OpenMed-Community/synthetic-neurology-conversations` directly from the Hugging Face Hub into a pandas `df` (single `messages` column, ~1.4k rows). This runs **before** Cell 1 so the rest of the pipeline consumes a real conversational neurology dataset. **This is the notebook only corpus — there is no synthetic fallback**, so Internet must be on and the load must succeed.

**Outputs:** global `df` (raw HF dataframe with one `messages` column).
**Requires:** Kaggle notebook with **Internet** enabled.


In [2]:
# ===== Neural Diagnosis - GraphRAG : PRE-CELL - Load HF dataset (REQUIRED) =====
!pip install -q datasets pyarrow

from datasets import load_dataset
import pandas as pd

# This notebook's ONLY corpus: OpenMed-Community/synthetic-neurology-conversations
# Requires Internet ON. There is no synthetic fallback; loading must succeed.
ds = load_dataset("OpenMed-Community/synthetic-neurology-conversations")
print(ds)

# Use only the train split, as the single source for the demo corpus.
df = ds["train"].to_pandas()

print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

assert isinstance(df, pd.DataFrame) and len(df) > 0, \
    "HF dataset did not load. Enable Internet and re-run the PRE-CELL."


README.md: 0.00B [00:00, ?B/s]

synthetic-neurology-conversations/train.(…):   0%|          | 0.00/4.38M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1452
    })
})
Shape: (1452, 1)


,messages
0,[{'content': 'What is the key characteristic o...
1,[{'content': 'What is the most common cause of...
2,[{'content': 'What is the significance of the ...
3,[{'content': 'How does the neurophysiological ...
4,[{'content': 'What are the two main components...


Columns: ['messages']


### Cell 1 — Parse the HF Conversational Dataset into the Corpus

**Purpose:** Take the Hugging Face dataset loaded by the PRE-CELL (`OpenMed-Community/synthetic-neurology-conversations`, single `messages` column) and flatten it into the unified corpus schema `{doc_id, title, text, source_type, source_priority, year, raw_path}` plus a turn-level view. Each conversation becomes one representative `conversation_text` document with `sample_id`, `turn_id`, `role`, `content`, `question`, `answer`, `source_type="synthetic_neurology_hf"`, `evidence_tier="synthetic_demo"`.

**This is the notebook's only corpus.** There is no synthetic/mock fallback; if `df` from the PRE-CELL is missing or empty, this cell raises an error.

**Inputs:** global `df` (pandas DataFrame from the PRE-CELL, `messages` column).
**Outputs:** `df_corpus`, `df_conversations` (turn-level view).
**Saved artifacts:** `WORK_DIR/artifacts/raw_corpus_preview.csv`, `WORK_DIR/artifacts/conversations_flattened.csv`.


In [3]:
# ===== Cell 1 - Parse HF conversations into the corpus (HF-only, no fallback) =====
HF_SOURCE = "OpenMed-Community/synthetic-neurology-conversations"

def _messages_col(df):
    """Return the messages column name (messages / conversation / conversations)."""
    cols_lower = {c.lower(): c for c in df.columns}
    return (cols_lower.get("messages") or cols_lower.get("conversation")
            or cols_lower.get("conversations"))

def _as_list(msgs):
    """Coerce an HF/pandas messages value into a plain python list.
    Handles python list/tuple, numpy array, and pandas-objet wrappers."""
    if msgs is None:
        return []
    if isinstance(msgs, (list, tuple)):
        return list(msgs)
    try:                       # numpy array or any ndarray-like
        return list(msgs.tolist() if hasattr(msgs, "tolist") else msgs)
    except TypeError:
        return []

def _parse_messages_column(df, msg_col):
    """Flatten the HF 'messages' column into turn-level + conversation-level tables.
    Returns (df_turns, df_repr).
      df_turns: one row per turn (sample_id, turn_id, role, content,
                conversation_text, question, answer, source_type, evidence_tier)
      df_repr : one row per conversation (title, text=conversation_text,
                source_type, year) so Cell 2/3/etc. treat each conversation as a doc.
    """
    turns_rows, conv_rows = [], []
    for i, msgs in enumerate(df[msg_col]):
        msgs = _as_list(msgs)
        if not msgs:
            continue
        conv_parts, first_user, first_asst = [], "", ""
        conv_turns = []
        for t_id, m in enumerate(msgs):
            if not isinstance(m, dict):
                continue
            role = m.get("role", "unknown")
            content = m.get("content") or m.get("text") or ""
            if not isinstance(content, str):
                content = (" ".join(str(x) for x in content)
                           if isinstance(content, (list, tuple)) else str(content))
            content = content.strip()
            if not content:
                continue
            conv_parts.append(f"{role}: {content}")
            if first_user == "" and role.lower() == "user":
                first_user = content
            if first_asst == "" and role.lower() == "assistant":
                first_asst = content
            row = {
                "sample_id": f"S{i:04d}",
                "turn_id": t_id,
                "role": role,
                "content": content,
                "source_type": "synthetic_neurology_hf",
                "evidence_tier": "synthetic_demo",
            }
            turns_rows.append(row)
            conv_turns.append(row)
        conversation_text = " || ".join(conv_parts) if conv_parts else ""
        for r in conv_turns:
            r["conversation_text"] = conversation_text
            r["question"] = first_user
            r["answer"] = first_asst
        title = (first_user[:60] + "...") if first_user else f"Conversation {i}"
        conv_rows.append({
            "text": conversation_text,
            "title": title,
            "source_type": "synthetic_neurology_hf",
            "year": np.nan,
            "conversation_text": conversation_text,
            "question": first_user,
            "answer": first_asst,
            "evidence_tier": "synthetic_demo",
        })
    empty_turns = ("sample_id", "turn_id", "role", "content", "conversation_text",
                    "question", "answer", "source_type", "evidence_tier")
    empty_repr = ("text", "title", "source_type", "year")
    df_turns = pd.DataFrame(turns_rows) if turns_rows else pd.DataFrame(columns=list(empty_turns))
    df_repr = pd.DataFrame(conv_rows) if conv_rows else pd.DataFrame(columns=list(empty_repr))
    return df_turns, df_repr

# ---- The HF dataset from the PRE-CELL is the ONLY corpus. ----
df_hf = globals().get("df")
if not isinstance(df_hf, pd.DataFrame) or len(df_hf) == 0:
    raise RuntimeError(
        "PRE-CELL did not produce a usable `df`. The HF dataset is the only corpus; "
        "enable Internet and re-run the PRE-CELL (load_dataset of " + HF_SOURCE + ").")

print(f"[load] HF dataset from PRE-CELL: rows={len(df_hf)}, cols={list(df_hf.columns)}")

# The HF dataset is conversational by construction; just locate the messages column.
_msg = _messages_col(df_hf)
if not _msg:
    raise RuntimeError(
        "Expected a 'messages' column from " + HF_SOURCE +
        "; got columns: " + str(list(df_hf.columns)))
df_conversations, df_repr = _parse_messages_column(df_hf, _msg)
if len(df_repr) == 0:
    raise RuntimeError("No conversations parsed from the HF 'messages' column.")

df_repr["raw_path"] = "huggingface:" + HF_SOURCE
df_corpus = df_repr.reset_index(drop=True)
used_external = True

# Canonical doc ids: one id per conversation
df_corpus.insert(0, "doc_id", ["D" + str(i).zfill(3) for i in range(len(df_corpus))])

print(f"[load] parsed {len(df_corpus)} conversations, {len(df_conversations)} turns.")
print("\n--- Corpus preview ---")
preview_cols = [c for c in ["doc_id", "source_type", "year", "title"] if c in df_corpus.columns]
print(df_corpus[preview_cols].to_string(index=False))
print(f"\nTotal docs: {len(df_corpus)} | HF dataset used: {used_external}")
if df_conversations is not None and len(df_conversations) > 0:
    print(f"Conversational dataset: {len(df_conversations)} flattened turns across "
          f"{df_conversations['sample_id'].nunique()} conversations.")
    print("Turn-level columns:", list(df_conversations.columns))
    df_conversations.head(5).to_csv(ART_DIR / "conversations_flattened.csv", index=False)
    print(f"Saved conversation preview -> {ART_DIR/'conversations_flattened.csv'}")
df_corpus.head(3).to_csv(ART_DIR / "raw_corpus_preview.csv", index=False)


[load] HF dataset from PRE-CELL: rows=1452, cols=['messages']
[load] parsed 1428 conversations, 5676 turns.

--- Corpus preview ---
doc_id            source_type  year                                                           title
  D000 synthetic_neurology_hf   NaN          What is the key characteristic of surface dyslexia?...
  D001 synthetic_neurology_hf   NaN                   What is the most common cause of dementia?...
  D002 synthetic_neurology_hf   NaN What is the significance of the median age of 83 years in th...
  D003 synthetic_neurology_hf   NaN How does the neurophysiological model differentiate between ...
  D004 synthetic_neurology_hf   NaN What are the two main components of the human nervous system...
  D005 synthetic_neurology_hf   NaN Describe the mechanism of the patellar tendon reflex (knee-j...
  D006 synthetic_neurology_hf   NaN What are some examples of visuospatial difficulties that ind...
  D007 synthetic_neurology_hf   NaN How does the gross appearance of

### Cell 2 — Cleaning, Deduplication & Evidence Ranking

**Purpose:** Clean whitespace/encoding, drop empty/broken rows, deduplicate by exact hash and
near-duplicate fuzzy similarity, then assign an **evidence priority rank**:
`guideline > systematic review > review > paper > unknown`.

**Inputs:** `df_corpus`.
**Outputs:** `df_clean` (cleaned, deduped, ranked corpus). Conversational (`synthetic_neurology_hf`)
rows are labeled `evidence_tier='synthetic_demo'` and given priority 1 so they rank below clinical
guidelines but still participate in retrieval/chunking for the demo pipeline.
**Saved artifacts:** `artifacts/corpus_clean.parquet`, `artifacts/corpus_clean.csv`.
**Customize later:** fuzzy threshold, additional source-type synonyms, custom priority map.


In [4]:
# ===== Cell 2 - Cleaning + dedup + evidence ranking =====
PRIORITY_MAP = {
    "guideline": 5, "guidelines": 5, "guidance": 5,
    "systematic review": 4, "systematic": 4, "meta-analysis": 4, "meta analysis": 4,
    "review": 3, "narrative review": 3,
    "paper": 2, "research": 2, "article": 2, "original": 2, "case report": 2, "case-report": 2,
    # --- Synthetic / conversational evidence ---
    "synthetic_neurology_hf": 1, "synthetic_neurology": 1,
    "synthetic_conversation": 1, "synthetic conversation": 1, "conversation": 1,
    "conversational": 1, "synthetic_demo": 1, "demo": 1, "synthetic": 1,
    "unknown": 1, "other": 1,
}

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    t = t.replace("\u2013", "-").replace("\u2014", "-")
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\n{2,}", " ", t)
    return t.strip()

def text_hash(t):
    return hashlib.md5(t.encode("utf-8")).hexdigest()

# Normalized fingerprint for near-duplicate detection (lowercase, alnum only)
def fingerprint(t):
    t = re.sub(r"[^a-z0-9 ]", " ", t.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t

def shingle_set(s, k=4):
    tokens = s.split()
    return set(" ".join(tokens[i:i+k]) for i in range(max(0, len(tokens)-k+1)))

def jaccard(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b); union = len(a | b)
    return inter / union if union else 0.0

def clean_and_dedup(df):
    d = df.copy()
    d["text"]  = d["text"].map(clean_text)
    d["title"] = d["title"].map(clean_text)
    d["source_type"] = d["source_type"].map(lambda x: clean_text(str(x)).lower())
    d = d[d["text"].str.len() > 30].reset_index(drop=True)

    # Exact-hash dedup
    d["hash"] = d["text"].map(text_hash)
    before = len(d)
    d = d.drop_duplicates(subset=["hash"]).reset_index(drop=True)
    print(f"[dedup] exact-hash: {before} -> {len(d)}")

    # Fuzzy near-duplicate dedup (keep higher-priority / earlier doc)
    d["fp"] = d["text"].map(fingerprint)
    d["shingles"] = d["fp"].map(lambda s: shingle_set(s, k=4))
    keep = [True] * len(d)
    THRESH = 0.85
    for i in range(len(d)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(d)):
            if not keep[j]:
                continue
            if jaccard(d.loc[i, "shingles"], d.loc[j, "shingles"]) >= THRESH:
                keep[j] = False  # drop later near-duplicate
    before2 = len(d)
    d = d[keep].reset_index(drop=True)
    print(f"[dedup] fuzzy(jaccard>={THRESH}): {before2} -> {len(d)}")

    # Evidence priority
    def priority(stype):
        stype = clean_text(str(stype)).lower()
        for k, v in PRIORITY_MAP.items():
            if k in stype:
                return v
        return PRIORITY_MAP["unknown"]
    d["source_priority"] = d["source_type"].map(priority)
    d["evidence_rank_label"] = d["source_priority"].map(
        {5: "guideline", 4: "systematic review", 3: "review", 2: "paper", 1: "unknown"})
    # Attach an explicit evidence_tier column if the source came in with one
    if "evidence_tier" in d.columns:
        d["evidence_tier"] = d["evidence_tier"].fillna(d["evidence_rank_label"])
    else:
        d["evidence_tier"] = d["evidence_rank_label"]
    # Relabel synthetic-conversation rows as 'demo' for clear communication
    d.loc[d["source_type"].str.contains(
        "synthetic_neurology_hf|synthetic_conversation|conversation", na=False),
        "evidence_rank_label"] = "demo"
    # Stable sort: priority desc, then year desc
    d = d.sort_values(["source_priority", "year"], ascending=[False, False]).reset_index(drop=True)
    d["doc_id"] = ["D" + str(i).zfill(3) for i in range(len(d))]
    return d.drop(columns=["hash", "fp", "shingles"])

df_clean = clean_and_dedup(df_corpus)

print("\n--- Evidence-type distribution ---")
print(df_clean["evidence_rank_label"].value_counts())
print("\n--- Cleaned corpus (top by priority) ---")
print(df_clean[["doc_id", "evidence_rank_label", "source_priority", "year", "title"]].head(10).to_string(index=False))

df_clean.to_parquet(ART_DIR / "corpus_clean.parquet", index=False)
df_clean.to_csv(ART_DIR / "corpus_clean.csv", index=False)
print(f"\nSaved cleaned corpus -> {ART_DIR/'corpus_clean.parquet'}")


[dedup] exact-hash: 1428 -> 1428
[dedup] fuzzy(jaccard>=0.85): 1428 -> 1428

--- Evidence-type distribution ---
evidence_rank_label
demo    1428
Name: count, dtype: int64

--- Cleaned corpus (top by priority) ---
doc_id evidence_rank_label  source_priority  year                                                           title
  D000                demo                1   NaN          What is the key characteristic of surface dyslexia?...
  D001                demo                1   NaN                   What is the most common cause of dementia?...
  D002                demo                1   NaN What is the significance of the median age of 83 years in th...
  D003                demo                1   NaN How does the neurophysiological model differentiate between ...
  D004                demo                1   NaN What are the two main components of the human nervous system...
  D005                demo                1   NaN Describe the mechanism of the patellar tendon reflex 

### Cell 3 — Semantic Chunking

**Purpose:** Split each cleaned doc into chunks that respect sentence boundaries and target size,
preserve full provenance (`doc_id`, `title`, `source_priority`, `year`, chunk index), and assign
stable `chunk_id`s.

**Inputs:** `df_clean`.
**Outputs:** `df_chunks`.
**Saved artifacts:** `artifacts/chunks.parquet`.
**Customize later:** chunk size/overlap, section-aware splitting.


In [5]:
# ===== Cell 3 - Semantic chunking =====
_SENT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

def split_sentences(text):
    text = clean_text(text)
    return [s for s in _SENT_RE.split(text) if s.strip()]

def chunk_doc(text, size=CONFIG["chunk_size"], overlap=CONFIG["chunk_overlap"]):
    sents = split_sentences(text)
    if not sents:
        return []
    chunks, cur, cur_len = [], [], 0
    for s in sents:
        cur.append(s); cur_len += len(s) + 1
        if cur_len >= size:
            chunks.append(" ".join(cur).strip())
            # carry overlap: keep last sentence(s) whose length <= overlap
            tail, tlen = [], 0
            for s2 in reversed(cur):
                if tlen + len(s2) > overlap:
                    break
                tail.append(s2); tlen += len(s2)
            cur = list(reversed(tail)); cur_len = tlen
    if cur:
        chunks.append(" ".join(cur).strip())
    return chunks

def build_chunks(df, size=None, overlap=None):
    size = size or CONFIG["chunk_size"]
    overlap = overlap or CONFIG["chunk_overlap"]
    rows = []
    for _, r in df.iterrows():
        # Choose the longest meaningful text for chunking:
        # conversation_text (built by Cell 1 for conversational datasets) if present, else r['text']
        src_text = r.get("conversation_text") or r["text"]
        parts = chunk_doc(src_text, size, overlap)
        if not parts:
            parts = [clean_text(src_text)]
        for idx, c in enumerate(parts):
            rows.append({
                "chunk_id": f"{r['doc_id']}_c{idx}",
                "doc_id": r["doc_id"],
                "chunk_idx": idx,
                "n_chunks_in_doc": len(parts),
                "text": c,
                "title": r["title"],
                "source_type": r["source_type"],
                "source_priority": r["source_priority"],
                "evidence_rank_label": r["evidence_rank_label"],
                "evidence_tier": r.get("evidence_tier", r["evidence_rank_label"]),
                "year": r["year"],
                "raw_path": r.get("raw_path", ""),
                # Carry forward conversational provenance when available
                "question": r.get("question", ""),
                "answer": r.get("answer", ""),
            })
    return pd.DataFrame(rows)

df_chunks = build_chunks(df_clean)
print(f"Total chunks: {len(df_chunks)} from {df_chunks['doc_id'].nunique()} docs")
print(f"Avg chars/chunk: {df_chunks['text'].str.len().mean():.0f}")
print("\nSample chunks:")
for _, r in df_chunks.head(3).iterrows():
    print(f"  [{r['chunk_id']}] ({r['evidence_rank_label']}) {r['text'][:140]}...")

df_chunks.to_parquet(ART_DIR / "chunks.parquet", index=False)
print(f"\nSaved chunks -> {ART_DIR/'chunks.parquet'}")


Total chunks: 12846 from 1428 docs
Avg chars/chunk: 530

Sample chunks:
  [D000_c0] (demo) user: What is the key characteristic of surface dyslexia? || assistant: The key characteristic of **surface dyslexia** is a selective impair...
  [D000_c1] (demo) Irregular items cannot be arrived at through "phonics first" routes; they must be stored as unique wholes in the orthographic lexicon. Inter...
  [D000_c2] (demo) 2. **Copy-Cover-Compare (CCC)** - Look: view the whole word. - Copy: reproduce it from memory while saying it. - Cover: hide the model and t...

Saved chunks -> /kaggle/working/artifacts/chunks.parquet


### Cell 4 — Medical Entity Normalization

**Purpose:** Extract neurology-relevant entities (symptoms, diagnoses, anatomy, imaging, labs, drugs,
procedures, contraindications, red flags) from chunks and normalize synonyms to canonical names
via a built-in **neurology lexicon**. SpaCy/scispaCy are used if available; otherwise a fast
lexicon-overlap matcher is used.

**Inputs:** `df_chunks`, `ENTITY_LEXICON`.
**Outputs:** `df_entities` (chunk-level entity table), normalized canonical terms.
**Saved artifacts:** `artifacts/entities.parquet`, `artifacts/entity_lexicon.json`.
**Customize later:** extend `ENTITY_LEXICON`, plug in scispaCy / SciSpacy UMLS linker.


In [6]:
# ===== Cell 4 - Medical entity normalization =====
# Neurology lexicon: canonical term -> list of surface variants. Categories drive KG typing.
ENTITY_LEXICON = {
    # --- Diagnoses ---
    "ischemic stroke":     {"category": "diagnosis", "variants": ["acute ischemic stroke", "ischaemic stroke", "stroke", "cerebral infarction"]},
    "subarachnoid hemorrhage": {"category": "diagnosis", "variants": ["sah", "subarachnoid haemorrhage", "ruptured aneurysm"]},
    "status epilepticus":  {"category": "diagnosis", "variants": ["continuous seizure", "recurrent seizures"]},
    "migraine":            {"category": "diagnosis", "variants": ["migraine with aura", "migraine headache"]},
    "bacterial meningitis":{"category": "diagnosis", "variants": ["meningococcal meningitis", "pneumococcal meningitis", "meningitis"]},
    "multiple sclerosis":  {"category": "diagnosis", "variants": ["ms", "demyelinating disease"]},
    "parkinson disease":   {"category": "diagnosis", "variants": ["parkinson's disease", "parkinsonism"]},
    "raised intracranial pressure": {"category": "diagnosis", "variants": ["raised icp", "increased intracranial pressure", "intracranial hypertension"]},
    "guillain-barre syndrome": {"category": "diagnosis", "variants": ["gbs", "acute ascending paralysis"]},
    "alzheimer disease":   {"category": "diagnosis", "variants": ["alzheimer's disease", "dementia"]},
    "transient ischemic attack": {"category": "diagnosis", "variants": ["tia", "mini-stroke"]},
    "bell palsy":          {"category": "diagnosis", "variants": ["facial nerve palsy", "idiopathic facial palsy"]},
    "trigeminal neuralgia":{"category": "diagnosis", "variants": ["tic douloureux"]},
    "subdural hematoma":   {"category": "diagnosis", "variants": ["subdural haematoma", "acute subdural", "chronic subdural"]},
    "myasthenia gravis":   {"category": "diagnosis", "variants": ["mg"]},
    "large vessel occlusion": {"category": "diagnosis", "variants": ["lvo"]},
    # --- Symptoms ---
    "facial droop":        {"category": "symptom", "variants": ["facial weakness"]},
    "headache":            {"category": "symptom", "variants": ["cephalgia"]},
    "thunderclap headache":{"category": "symptom", "variants": ["sudden severe headache", "worst headache"]},
    "seizure":             {"category": "symptom", "variants": ["convulsion", "epileptic seizure"]},
    "neck stiffness":      {"category": "symptom", "variants": ["nuchal rigidity", "meningismus"]},
    "photophobia":         {"category": "symptom", "variants": []},
    "phonophobia":         {"category": "symptom", "variants": []},
    "altered consciousness":{"category": "symptom", "variants": ["altered mental status", "confusion"]},
    "resting tremor":      {"category": "symptom", "variants": ["tremor"]},
    "bradykinesia":        {"category": "symptom", "variants": []},
    "rigidity":            {"category": "symptom", "variants": []},
    "ptosis":              {"category": "symptom", "variants": []},
    "diplopia":            {"category": "symptom", "variants": ["double vision"]},
    "papilledema":         {"category": "symptom", "variants": []},
    "flaccid paralysis":   {"category": "symptom", "variants": ["ascending paralysis", "weakness"]},
    "memory loss":         {"category": "symptom", "variants": ["cognitive decline"]},
    "vomiting":            {"category": "symptom", "variants": ["emesis"]},
    "fever":               {"category": "symptom", "variants": ["pyrexia"]},
    "petechial rash":      {"category": "symptom", "variants": ["petechiae"]},
    # --- Anatomy ---
    "brain":               {"category": "anatomy", "variants": ["cerebrum"]},
    "hippocampus":         {"category": "anatomy", "variants": []},
    "trigeminal nerve":    {"category": "anatomy", "variants": ["fifth nerve"]},
    "facial nerve":        {"category": "anatomy", "variants": ["seventh nerve"]},
    "cerebellopontine angle": {"category": "anatomy", "variants": ["cpa"]},
    # --- Imaging / procedures ---
    "non-contrast ct":     {"category": "imaging", "variants": ["non contrast ct", "ncct", "head ct", "ct head", "ct brain"]},
    "mri":                 {"category": "imaging", "variants": ["magnetic resonance imaging", "mri brain"]},
    "lumbar puncture":     {"category": "procedure", "variants": ["spinal tap", "lp"]},
    "carotid imaging":     {"category": "imaging", "variants": ["carotid ultrasound", "carotid doppler"]},
    "echocardiography":    {"category": "imaging", "variants": ["echo", "tte"]},
    "nerve conduction study": {"category": "procedure", "variants": ["ncs", "nerve conduction"]},
    "electromyography":    {"category": "procedure", "variants": ["emg"]},
    "mechanical thrombectomy": {"category": "procedure", "variants": ["thrombectomy", "endovascular clot retrieval"]},
    "surgical evacuation": {"category": "procedure", "variants": ["craniotomy", "evacuation"]},
    "microvascular decompression": {"category": "procedure", "variants": ["mvd"]},
    "thymectomy":          {"category": "procedure", "variants": []},
    # --- Labs ---
    "xanthochromia":       {"category": "lab", "variants": []},
    "csf oligoclonal bands":{"category": "lab", "variants": ["oligoclonal bands", "ocb"]},
    "albuminocytologic dissociation": {"category": "lab", "variants": []},
    "acetylcholine receptor antibodies": {"category": "lab", "variants": ["achr antibodies", "achr antibody"]},
    # --- Drugs ---
    "alteplase":           {"category": "drug", "variants": ["tissue plasminogen activator", "tpa", "thrombolysis"]},
    "lorazepam":           {"category": "drug", "variants": []},
    "diazepam":            {"category": "drug", "variants": []},
    "midazolam":           {"category": "drug", "variants": []},
    "fosphenytoin":        {"category": "drug", "variants": []},
    "valproate":           {"category": "drug", "variants": ["valproic acid", "sodium valproate"]},
    "levetiracetam":       {"category": "drug", "variants": []},
    "sumatriptan":         {"category": "drug", "variants": ["triptans"]},
    "propranolol":         {"category": "drug", "variants": []},
    "topiramate":          {"category": "drug", "variants": []},
    "amitriptyline":       {"category": "drug", "variants": []},
    "ceftriaxone":         {"category": "drug", "variants": []},
    "vancomycin":          {"category": "drug", "variants": []},
    "dexamethasone":       {"category": "drug", "variants": []},
    "interferon-beta":     {"category": "drug", "variants": ["interferon beta", "ifn-beta"]},
    "glatiramer acetate":  {"category": "drug", "variants": []},
    "natalizumab":         {"category": "drug", "variants": []},
    "ocrelizumab":         {"category": "drug", "variants": []},
    "levodopa":            {"category": "drug", "variants": ["l-dopa", "carbidopa-levodopa"]},
    "carbidopa":           {"category": "drug", "variants": []},
    "pramipexole":         {"category": "drug", "variants": ["dopamine agonist"]},
    "rasagiline":          {"category": "drug", "variants": ["mao-b inhibitor"]},
    "mannitol":            {"category": "drug", "variants": []},
    "hypertonic saline":   {"category": "drug", "variants": []},
    "intravenous immunoglobulin": {"category": "drug", "variants": ["ivig"]},
    "plasma exchange":     {"category": "drug", "variants": ["plasmapheresis", "plex"]},
    "donepezil":           {"category": "drug", "variants": []},
    "rivastigmine":        {"category": "drug", "variants": []},
    "galantamine":         {"category": "drug", "variants": []},
    "memantine":           {"category": "drug", "variants": []},
    "aspirin":             {"category": "drug", "variants": []},
    "statin":              {"category": "drug", "variants": ["statins"]},
    "prednisolone":        {"category": "drug", "variants": ["prednisone", "corticosteroid"]},
    "acyclovir":           {"category": "drug", "variants": ["aciclovir"]},
    "carbamazepine":       {"category": "drug", "variants": []},
    "oxcarbazepine":       {"category": "drug", "variants": []},
    # --- Red flags ---
    "thunderclap headache": {"category": "red_flag", "variants": []},
    "cushing triad":       {"category": "red_flag", "variants": []},
    "midline shift":       {"category": "red_flag", "variants": ["mass effect"]},
    "diaphragmatic failure":{"category": "red_flag", "variants": ["respiratory failure"]},
    # --- Contraindications ---
    "hemorrhage":          {"category": "contraindication", "variants": ["haemorrhage", "bleeding"]},
}

def build_matcher(lexicon):
    """term(lowercased) -> (canonical, category). Longest variant first for greedy matching."""
    pairs = []
    for canon, info in lexicon.items():
        pairs.append((canon.lower(), canon, info["category"]))
        for v in info.get("variants", []):
            pairs.append((v.lower(), canon, info["category"]))
    pairs.sort(key=lambda x: -len(x[0]))  # longest first
    return pairs

_MATCHER = build_matcher(ENTITY_LEXICON)

def extract_entities(text):
    """Lexicon-overlap entity extractor with canonical normalization."""
    low = " " + clean_text(text).lower() + " "
    found = []
    seen_spans = []
    for surface, canon, cat in _MATCHER:
        # word-boundary search
        pat = r"(?<![a-z0-9])" + re.escape(surface) + r"(?![a-z0-9])"
        for m in re.finditer(pat, low):
            s, e = m.start(), m.end()
            # skip if span inside an already-matched longer span
            if any(s >= ss and e <= es for ss, es in seen_spans):
                continue
            seen_spans.append((s, e))
            found.append({"surface": low[s:e].strip(), "canonical": canon, "category": cat, "span": (s, e)})
    # de-dup keeping canonical+category
    uniq = {}
    for f in found:
        key = (f["canonical"], f["category"])
        if key not in uniq:
            uniq[key] = f
    return list(uniq.values())

# Optional: scispaCy enhancement if installed (purely additive)
try:
    import spacy
    _NLP = spacy.load("en_core_sci_sm") if spacy.util.is_package("en_core_sci_sm") else None
except Exception:
    _NLP = None
if _NLP is None:
    print("[info] scispaCy not available - using lexicon-overlap entity extractor (notebook-friendly default).")

def extract_entities_hybrid(text):
    base = extract_entities(text)
    canon_set = {(b["canonical"], b["category"]) for b in base}
    if _NLP is not None:
        doc = _NLP(text)
        for ent in doc.ents:
            key = (ent.text.lower(), "ner")
            # only add if not already covered canonically (keeps provenance simple)
            base.append({"surface": ent.text, "canonical": ent.text.lower(), "category": "ner", "span": (ent.start_char, ent.end_char)})
    return base

rows = []
for _, r in df_chunks.iterrows():
    ents = extract_entities_hybrid(r["text"])
    for e in ents:
        rows.append({
            "chunk_id": r["chunk_id"], "doc_id": r["doc_id"],
            "canonical": e["canonical"], "category": e["category"],
            "surface": e["surface"],
            "source_priority": r["source_priority"],
            "evidence_rank_label": r["evidence_rank_label"],
        })
df_entities = pd.DataFrame(rows)

print(f"Total entity mentions: {len(df_entities)}")
print(f"Unique canonical entities: {df_entities['canonical'].nunique()}")
print("\nEntity count by category:")
print(df_entities["category"].value_counts())
print("\nTop entities:")
print(df_entities["canonical"].value_counts().head(12))

df_entities.to_parquet(ART_DIR / "entities.parquet", index=False)
with open(ART_DIR / "entity_lexicon.json", "w") as f:
    json.dump(ENTITY_LEXICON, f, indent=2)
print(f"\nSaved entities + lexicon -> {ART_DIR}")


[info] scispaCy not available - using lexicon-overlap entity extractor (notebook-friendly default).
Total entity mentions: 5643
Unique canonical entities: 86

Entity count by category:
category
diagnosis           2078
anatomy             1508
symptom              929
imaging              529
drug                 277
procedure            205
contraindication      58
lab                   33
red_flag              26
Name: count, dtype: int64

Top entities:
canonical
brain                 1292
alzheimer disease      765
mri                    495
multiple sclerosis     430
ischemic stroke        420
flaccid paralysis      306
hippocampus            196
memory loss            180
myasthenia gravis      166
parkinson disease      157
electromyography       113
seizure                 82
Name: count, dtype: int64

Saved entities + lexicon -> /kaggle/working/artifacts


### Cell 5 — Neurology Knowledge-Graph Construction

**Purpose:** Build a networkx MultiDiGraph of canonical entities (nodes, typed by category) and
typed, **provenance-preserving** edges. Relations are inferred from simple, transparent
co-occurrence + category-pair rules (e.g. symptom→`suggests`→diagnosis, diagnosis→`confirmed_by`→imaging/procedure/lab,
diagnosis→`treated_by`→drug, drug→`contraindicated_in`→contraindication, diagnosis→`recommended_by`→guideline).

**Inputs:** `df_chunks`, `df_entities`, `ENTITY_LEXICON`.
**Outputs:** `G` (networkx graph), `df_edges`, `df_nodes`.
**Saved artifacts:** `artifacts/kg_nodes.parquet`, `artifacts/kg_edges.parquet`, `artifacts/kg.gpickle`.
**Customize later:** add explicit relation templates, GNN-based link prediction, UMLS/CUI linking.


In [7]:
# ===== Cell 5 - Neurology KG construction =====
# Category-pair -> relation inference rules. (src_cat, tgt_cat) -> relation
RELATION_RULES = [
    (("symptom", "diagnosis"), "suggests"),
    (("diagnosis", "symptom"), "presents_with"),
    (("diagnosis", "imaging"), "confirmed_by"),
    (("diagnosis", "procedure"), "confirmed_by"),
    (("diagnosis", "lab"), "confirmed_by"),
    (("diagnosis", "drug"), "treated_by"),
    (("drug", "diagnosis"), "treats"),
    (("drug", "contraindication"), "contraindicated_in"),
    (("contraindication", "drug"), "contraindicates"),
    (("diagnosis", "diagnosis"), "differential_of"),
    (("symptom", "symptom"), "associated_with"),
    (("diagnosis", "anatomy"), "located_in"),
    (("drug", "drug"), "associated_with"),
    (("imaging", "diagnosis"), "supports"),
    (("lab", "diagnosis"), "supports"),
    (("procedure", "diagnosis"), "supports"),
    (("red_flag", "diagnosis"), "risk_for"),
    (("diagnosis", "red_flag"), "warning_sign"),
]
RULE_LOOKUP = {pair: rel for pair, rel in RELATION_RULES}

# Evidence-priority -> weight, used by graph support scoring later
PRIORITY_WEIGHT = {5: 1.0, 4: 0.9, 3: 0.75, 2: 0.6, 1: 0.4}

def build_kg(df_chunks, df_entities):
    G = nx.MultiDiGraph()
    # nodes
    cat_of = {}
    for canon, grp in df_entities.groupby("canonical"):
        cat = grp["category"].mode().iat[0]
        cat_of[canon] = cat
        G.add_node(canon, category=cat,
                   n_chunks=int(grp["chunk_id"].nunique()),
                   n_docs=int(grp["doc_id"].nunique()),
                   max_priority=int(grp["source_priority"].max()))
    # co-occurrence within a chunk -> candidate edges
    edge_buf = defaultdict(list)  # (src, tgt, rel) -> list of provenance dicts
    for chunk_id, grp in df_entities.groupby("chunk_id"):
        canons = grp["canonical"].unique().tolist()
        cats = {c: cat_of.get(c, "unknown") for c in canons}
        pri = int(grp["source_priority"].iloc[0])
        label = grp["evidence_rank_label"].iloc[0]
        doc_id = grp["doc_id"].iloc[0]
        for i in range(len(canons)):
            for j in range(len(canons)):
                if i == j:
                    continue
                a, b = canons[i], canons[j]
                ca, cb = cats[a], cats[b]
                rel = RULE_LOOKUP.get((ca, cb))
                if rel is None:
                    continue
                key = (a, b, rel)
                edge_buf[key].append({
                    "chunk_id": chunk_id, "doc_id": doc_id,
                    "source_priority": pri, "evidence_rank_label": label,
                    "weight": PRIORITY_WEIGHT.get(pri, 0.4),
                })
    # commit edges, aggregating provenance
    rows = []
    for (a, b, rel), provs in edge_buf.items():
        w = sum(p["weight"] for p in provs)
        G.add_edge(a, b, relation=rel, weight=w,
                   n_occurrences=len(provs),
                   provenance=provs,
                   chunk_ids=list({p["chunk_id"] for p in provs}),
                   doc_ids=list({p["doc_id"] for p in provs}),
                   max_priority=max(p["source_priority"] for p in provs))
        rows.append({
            "src": a, "tgt": b, "relation": rel, "weight": round(w, 3),
            "n_occurrences": len(provs),
            "max_priority": max(p["source_priority"] for p in provs),
            "n_chunks": len({p["chunk_id"] for p in provs}),
            "example_chunk": provs[0]["chunk_id"],
        })
    df_edges = pd.DataFrame(rows).sort_values("weight", ascending=False).reset_index(drop=True)
    df_nodes = pd.DataFrame([
        {"node": n, "category": d["category"], "n_chunks": d["n_chunks"],
         "n_docs": d["n_docs"], "max_priority": d["max_priority"]}
        for n, d in G.nodes(data=True)
    ]).sort_values(["n_chunks", "n_docs"], ascending=False).reset_index(drop=True)
    return G, df_edges, df_nodes

G, df_edges, df_nodes = build_kg(df_chunks, df_entities)
print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("\nNodes by category:")
print(df_nodes["category"].value_counts())
print("\nTop edges:")
print(df_edges.head(12).to_string(index=False))
print("\nRelation distribution:")
print(df_edges["relation"].value_counts())

df_nodes.to_parquet(ART_DIR / "kg_nodes.parquet", index=False)
df_edges.to_parquet(ART_DIR / "kg_edges.parquet", index=False)
with open(ART_DIR / "kg.gpickle", "wb") as f:
    pickle.dump(G, f)
print(f"\nSaved KG -> {ART_DIR}")


KG: 86 nodes, 940 edges

Nodes by category:
category
drug                29
symptom             17
diagnosis           16
procedure            7
anatomy              5
imaging              4
lab                  4
red_flag             3
contraindication     1
Name: count, dtype: int64

Top edges:
               src                tgt        relation  weight  n_occurrences  max_priority  n_chunks example_chunk
   ischemic stroke              brain      located_in    41.6            104             1       104       D035_c2
       memory loss  alzheimer disease        suggests    39.6             99             1        99       D006_c0
 alzheimer disease        memory loss   presents_with    39.6             99             1        99       D006_c0
 alzheimer disease              brain      located_in    37.6             94             1        94       D011_c1
   ischemic stroke  alzheimer disease differential_of    32.4             81             1        81       D010_c6
 alzheimer d

### Cell 6 — Embeddings Generation

**Purpose:** Build a **pluggable** embedder. Default = `sentence-transformers/all-MiniLM-L6-v2` if available;
fallback = TF-IDF + SVD hashing to a fixed dim. Pluggable for Gemini/Groq later via `API_KEYS`.

**Inputs:** `df_chunks`.
**Outputs:** `embedder`, `chunk_embeddings` (np.ndarray [N, D]).
**Saved artifacts:** `artifacts/chunk_embeddings.npy`.
**Customize later:** swap to a domain model (`pritamdeka/S-PubMedBert-MS-MARCO`) or an API embedder.


In [8]:
# ===== Cell 6 - Embeddings generation =====
class Embedder:
    """Pluggable embedder: sentence-transformers if available, else TF-IDF/SVD fallback.
    TODO(later): add Gemini/Groq backends gated on API_KEYS."""
    def __init__(self, model_name=CONFIG["embed_model_name"], dim=CONFIG["embed_dim"]):
        self.model_name = model_name
        self.dim = dim
        self.backend = "st" if HAVE_ST else "tfidf"
        self._st = None
        self._tv = None
        self._svd = None
        if self.backend == "st":
            try:
                self._st = SentenceTransformer(model_name)
                # update true dim from model
                test = self._st.encode(["probe"], normalize_embeddings=True)
                self.dim = int(test.shape[1])
                print(f"[embedder] sentence-transformers '{model_name}' loaded, dim={self.dim}")
            except Exception as e:
                print(f"[embedder] ST load failed ({e}); falling back to TF-IDF/SVD.")
                self.backend = "tfidf"
        if self.backend == "tfidf":
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.decomposition import TruncatedSVD
            self._tv = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True, stop_words="english")
            self._svd = TruncatedSVD(n_components=min(dim, 256), random_state=42)
            print(f"[embedder] TF-IDF + SVD fallback, dim={self._svd.n_components}")

    def _norm(self, v):
        n = np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
        return v / n

    def fit(self, texts):
        if self.backend == "tfidf":
            X = self._tv.fit_transform(texts)
            self._svd.fit(X)
        return self

    def encode(self, texts, batch_size=64):
        if self.backend == "st":
            v = self._st.encode(list(texts), batch_size=batch_size,
                                show_progress_bar=False, normalize_embeddings=True)
            return np.asarray(v, dtype="float32")
        else:
            X = self._tv.transform(texts)
            v = self._svd.transform(X).astype("float32")
            return self._norm(v)

embedder = Embedder().fit(df_chunks["text"].tolist())
chunk_embeddings = embedder.encode(df_chunks["text"].tolist())
print(f"chunk_embeddings shape: {chunk_embeddings.shape}")
# quick sanity: nearest neighbor of chunk 0
sims = chunk_embeddings @ chunk_embeddings[0]
top = np.argsort(-sims)[1:4]
print("Sanity - closest chunks to chunk 0:")
for i in top:
    print(f"  sim={sims[i]:.3f} [{df_chunks.iloc[i]['chunk_id']}] {df_chunks.iloc[i]['text'][:90]}...")
np.save(ART_DIR / "chunk_embeddings.npy", chunk_embeddings)
print(f"\nSaved embeddings -> {ART_DIR/'chunk_embeddings.npy'}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[embedder] sentence-transformers 'sentence-transformers/all-MiniLM-L6-v2' loaded, dim=384
chunk_embeddings shape: (12846, 384)
Sanity - closest chunks to chunk 0:
  sim=0.820 [D735_c4] In summary, **surface dyslexia is characterized by an overreliance on phonological rules**...
  sim=0.794 [D1348_c2] 2. Central dyslexias - the visual word form reaches the reading system, but the mapping to...
  sim=0.780 [D222_c0] user: What is the primary difficulty experienced by individuals with phonological dyslexia...

Saved embeddings -> /kaggle/working/artifacts/chunk_embeddings.npy


### Cell 7 — Build FAISS Index

**Purpose:** Build a normalized inner-product (cosine) FAISS index over chunk embeddings with a
numpy-cosine fallback when FAISS isn't installed. Maintain id-mapping metadata.

**Inputs:** `chunk_embeddings`, `df_chunks`.
**Outputs:** `faiss_index` (or fallback object), `chunk_ids`, `idx_to_chunk`.
**Saved artifacts:** `artifacts/faiss.index` (when FAISS present), `artifacts/faiss_meta.json`.
**Customize later:** IVF/HNSW for very large corpora, GPU index, external vector DB.


In [9]:
# ===== Cell 7 - Build FAISS index =====
chunk_ids = df_chunks["chunk_id"].tolist()
idx_to_chunk = {i: cid for i, cid in enumerate(chunk_ids)}
chunk_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}
D = chunk_embeddings.shape[1]

class NumpyCosIndex:
    """Fallback 'FAISS-like' index: stores normalized vectors, returns (scores, indices)."""
    def __init__(self, vectors):
        self.vectors = np.asarray(vectors, dtype="float32")
    def search(self, q, k):
        q = np.asarray(q, dtype="float32")
        if q.ndim == 1:
            q = q[None, :]
        scores = (q @ self.vectors.T)[0]
        idx = np.argsort(-scores)[:k]
        return scores[idx][None, :], idx[None, :]

if HAVE_FAISS:
    faiss_index = faiss.IndexFlatIP(D)
    faiss_index.add(np.ascontiguousarray(chunk_embeddings.astype("float32")))
    faiss.write_index(faiss_index, str(ART_DIR / "faiss.index"))
    print(f"[faiss] IndexFlatIP built with {faiss_index.ntotal} vectors, dim={D}")
else:
    faiss_index = NumpyCosIndex(chunk_embeddings)
    print(f"[numpy] cosine fallback index built with {len(chunk_ids)} vectors, dim={D}")

with open(ART_DIR / "faiss_meta.json", "w") as f:
    json.dump({"chunk_ids": chunk_ids, "dim": int(D), "backend": "faiss" if HAVE_FAISS else "numpy"}, f)

def dense_search(query_vec, k=CONFIG["top_k_hybrid"]):
    # Ensure 2-D (n, D): FAISS requires it; HybridRetriever passes a 1-D vector.
    q = np.asarray(query_vec, dtype="float32")
    if q.ndim == 1:
        q = q[None, :]
    scores, idx = faiss_index.search(np.ascontiguousarray(q), k)
    return [(idx_to_chunk[int(i)], float(scores[0][p])) for p, i in enumerate(idx[0])]
# quick test
_qv = embedder.encode(["sudden severe headache and stiff neck"])
print("\nDense test 'sudden severe headache and stiff neck':")
for cid, sc in dense_search(_qv, k=5):
    print(f"  {sc:.3f} [{cid}] {df_chunks.loc[df_chunks.chunk_id==cid,'text'].iat[0][:90]}...")


[faiss] IndexFlatIP built with 12846 vectors, dim=384

Dense test 'sudden severe headache and stiff neck':
  0.501 [D1338_c3] 3. Key procedural considerations • Contra-indications: Signs of raised intracranial pressu...
  0.475 [D235_c2] 1. Generalized tetanus (≈ 80 % of cases) • Prodrome: malaise, headache, jaw stiffness or s...
  0.474 [D1185_c2] 2. **Detailed Descriptions of Neurological Symptoms** - **Epilepsy**: Differentiated **epi...
  0.469 [D789_c5] Key early signs of dysfunction to watch for: * **Unilateral facial weakness or paralysis**...
  0.466 [D518_c2] Headache and facial pain syndromes • Migraine (with/without aura, chronic migraine) • Tens...


### Cell 8 — BM25 / Sparse Retrieval

**Purpose:** Build a sparse retriever. Uses `rank_bm25` if available; otherwise a TF-IDF cosine
fallback that exposes the same `.search(query, k)` API. Tokenization lowercases, strips non-alnum,
keeps medical multi-word handling reasonable.

**Inputs:** `df_chunks`.
**Outputs:** `sparse_retriever`.
**Saved artifacts:** `artifacts/bm25.pkl`.
**Customize later:** domain stopwords, BM25 params (k1/b), SPLADE/BGE-M3.


In [10]:
# ===== Cell 8 - BM25 sparse retrieval =====
_TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text):
    return _TOKEN_RE.findall(clean_text(text).lower())

class BM25Retriever:
    def __init__(self, docs):
        self.chunk_ids = [d["chunk_id"] for d in docs]
        self.tokenized = [tokenize(d["text"]) for d in docs]
        if HAVE_BM25:
            self.bm25 = BM25Okapi(self.tokenized)
            self.backend = "bm25"
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.metrics.pairwise import cosine_similarity
            self._tv = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)
            self._matrix = self._tv.fit_transform([d["text"] for d in docs])
            self._cos = cosine_similarity
            self.backend = "tfidf-cosine"
        print(f"[sparse] backend={self.backend}, n_docs={len(self.chunk_ids)}")

    def search(self, query, k=CONFIG["top_k_hybrid"]):
        if self.backend == "bm25":
            scores = np.asarray(self.bm25.get_scores(tokenize(query)), dtype="float32")
        else:
            qv = self._tv.transform([query])
            scores = (self._matrix @ qv.T).toarray().ravel()
        # min-max normalize scores to [0,1] for stable fusion
        if scores.size == 0:
            return []
        smin, smax = scores.min(), scores.max()
        norm = (scores - smin) / (smax - smin + 1e-9)
        order = np.argsort(-norm)[:k]
        return [(self.chunk_ids[i], float(norm[i])) for i in order]

sparse_retriever = BM25Retriever(df_chunks[["chunk_id", "text"]].to_dict("records"))
print("Test 'status epilepticus first line treatment':")
for cid, sc in sparse_retriever.search("status epilepticus first line treatment", k=5):
    print(f"  {sc:.3f} [{cid}] {df_chunks.loc[df_chunks.chunk_id==cid,'text'].iat[0][:90]}...")
with open(ART_DIR / "bm25.pkl", "wb") as f:
    pickle.dump({"chunk_ids": sparse_retriever.chunk_ids, "backend": sparse_retriever.backend}, f)


[sparse] backend=bm25, n_docs=12846
Test 'status epilepticus first line treatment':
  1.000 [D522_c2] 4. Emergency Medicine - Acute stroke triage, status epilepticus, thunderclap headache, acu...
  0.919 [D518_c1] Cerebrovascular disease • Ischemic stroke / transient ischemic attack (TIA) • Intracerebra...
  0.886 [D794_c9] Epilepsy ↔ Movement Disorders ↔ Neuro-critical Care The same intracranial electrodes used ...
  0.859 [D290_c0] user: When should antidepressant medications be considered in the treatment of dementia? |...
  0.853 [D957_c4] In the largest case-series and national registries the lag to correct diagnosis ranges fro...


### Cell 9 — Hybrid (Dense + Sparse) Retrieval

**Purpose:** Fuse dense FAISS and sparse BM25 scores via convex combination
(`dense_weight`, `sparse_weight`) and return a single ranked candidate list for reranking.

**Inputs:** `embedder`, `faiss_index`, `sparse_retriever`, `df_chunks`.
**Outputs:** `hybrid_retriever` with `.search(query, k)`.
**Saved artifacts:** none (in-memory).
**Customize later:** RRF (Reciprocal Rank Fusion) alternative, weight tuning.


In [11]:
# ===== Cell 9 - Hybrid retrieval =====
class HybridRetriever:
    def __init__(self, embedder, dense_index_search, sparse_search, chunk_meta,
                 dw=CONFIG["dense_weight"], sw=CONFIG["sparse_weight"]):
        self.embedder = embedder
        self.dense_search = dense_index_search   # function(query_vec, k)
        self.sparse_search = sparse_search       # function(query, k)
        self.meta = chunk_meta                   # dict chunk_id -> row dict
        self.dw, self.sw = dw, sw

    def search(self, query, k=CONFIG["top_k_hybrid"]):
        qv = self.embedder.encode([query])[0]
        # over-fetch to give fusion a fair pool
        d = dict(self.dense_search(qv, k=k*3))
        s = dict(self.sparse_search(query, k=k*3))
        candidates = set(d) | set(s)
        scored = []
        for cid in candidates:
            ds = float(d.get(cid, 0.0)); ss = float(s.get(cid, 0.0))
            fused = self.dw * ds + self.sw * ss
            scored.append((cid, fused, ds, ss))
        scored.sort(key=lambda x: -x[1])
        results = []
        for cid, fused, ds, ss in scored[:k]:
            m = self.meta[cid]
            results.append({
                "chunk_id": cid, "fused_score": fused,
                "dense_score": ds, "sparse_score": ss,
                "text": m["text"], "title": m["title"],
                "doc_id": m["doc_id"], "source_priority": m["source_priority"],
                "evidence_rank_label": m["evidence_rank_label"], "year": m["year"],
            })
        return results

chunk_meta = {r["chunk_id"]: r for _, r in df_chunks.iterrows()}
hybrid_retriever = HybridRetriever(embedder, dense_search, sparse_retriever.search, chunk_meta)

print("Hybrid test 'raised intracranial pressure emergency management':")
for r in hybrid_retriever.search("raised intracranial pressure emergency management", k=5):
    print(f"  fused={r['fused_score']:.3f} (d={r['dense_score']:.2f}, s={r['sparse_score']:.2f}) "
          f"[{r['chunk_id']}] {r['evidence_rank_label']} :: {r['text'][:80]}...")


Hybrid test 'raised intracranial pressure emergency management':
  fused=0.696 (d=0.49, s=1.00) [D1320_c2] demo :: In dynamic terms, the CSF transmits intracranial pressure evenly, while the dura...
  fused=0.692 (d=0.54, s=0.92) [D1338_c3] demo :: 3. Key procedural considerations • Contra-indications: Signs of raised intracran...
  fused=0.631 (d=0.55, s=0.75) [D1091_c0] demo :: user: What types of care are often required following acute neurological events ...
  fused=0.556 (d=0.57, s=0.53) [D1185_c3] demo :: 3. **Neuroanatomical Insights** - Followed **Vesalian anatomy**, correcting medi...
  fused=0.545 (d=0.49, s=0.63) [D1338_c2] demo :: 2. Therapeutic and investigative roles • Removal of CSF to relieve pressure in i...


### Cell 10 — Graph / Path Retrieval

**Purpose:** Extract entities from the query, locate them in the KG, and return relevant nodes,
their 1–2 hop neighbors, and short paths between query entities — each piece of evidence carrying
**provenance** (chunk_ids / doc_ids / priority) used later by the graph-aware reranker.

**Inputs:** `G`, `df_edges`, entity extractor.
**Outputs:** `graph_retriever` with `.retrieve(query, k)`.
**Saved artifacts:** none (in-memory).
**Customize later:** personalization-weighted random walks, k-hop with decay, GNN node embeddings.


In [12]:
# ===== Cell 10 - Graph / path retrieval =====
class GraphRetriever:
    def __init__(self, G, df_edges, df_entities, extract_fn, chunk_meta):
        self.G = G
        self.df_edges = df_edges
        self.df_entities = df_entities
        self.extract_fn = extract_fn
        self.meta = chunk_meta
        # index: canonical entity -> set(chunk_id)
        self.ent_to_chunks = df_entities.groupby("canonical")["chunk_id"].apply(lambda s: set(s)).to_dict()

    def _query_entities(self, query):
        ents = self.extract_fn(query)
        canon = [e["canonical"] for e in ents]
        canon = [c for c in canon if c in self.G]
        # de-dup preserving order
        seen, out = set(), []
        for c in canon:
            if c not in seen:
                seen.add(c); out.append(c)
        return out

    def retrieve(self, query, max_hops=2, top_paths=8):
        q_ents = self._query_entities(query)
        if not q_ents:
            return {"query_entities": [], "nodes": [], "paths": [], "chunk_ids": [], "graph_score": {}}

        # 1) collect neighbor nodes within max_hops with decay
        node_scores = defaultdict(float)
        node_prov = defaultdict(set)  # node -> chunk_ids
        for src in q_ents:
            if src not in self.G:
                continue
            for hop in range(1, max_hops + 1):
                for n in nx.single_source_shortest_path_length(self.G, src, cutoff=hop):
                    if n == src:
                        continue
                    decay = 1.0 / hop
                    # weight by best edge weight into n
                    best_w = 0.0
                    best_chunks = set()
                    # undirected view: check both directions across multiedges
                    for u, v, ed in self.G.edges(data=True):
                        if (u == src and v == n) or (u == n and v == src):
                            if ed["weight"] > best_w:
                                best_w = ed["weight"]
                                best_chunks = set(ed.get("chunk_ids", []))
                    node_scores[n] += decay * best_w
                    node_prov[n] |= best_chunks
                    node_prov[n] |= self.ent_to_chunks.get(n, set())

        # 2) short paths between pairs of query entities
        paths = []
        for i in range(len(q_ents)):
            for j in range(len(q_ents)):
                if i == j:
                    continue
                a, b = q_ents[i], q_ents[j]
                try:
                    for p in nx.all_simple_paths(self.G.to_undirected(), a, b, cutoff=4):
                        paths.append(p)
                        if len(paths) >= top_paths * 3:
                            break
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    pass
        # rank paths by sum of edge weights
        def path_weight(p):
            w = 0.0
            und = self.G.to_undirected()
            for u, v in zip(p, p[1:]):
                md = und.get_edge_data(u, v)
                if md:
                    w += max(d.get("weight", 0) for d in md.values())
            return w
        paths = sorted(set(tuple(p) for p in paths), key=path_weight, reverse=True)[:top_paths]

        # 3) collect chunk_ids touched by these nodes/paths, with a graph support score
        graph_score = defaultdict(float)
        for node, sc in node_scores.items():
            for cid in node_prov.get(node, set()):
                graph_score[cid] += sc
        for p in paths:
            und = self.G.to_undirected()
            for u, v in zip(p, p[1:]):
                md = und.get_edge_data(u, v)
                if md:
                    w = max(d.get("weight", 0) for d in md.values())
                    for cid in (set().union(*[d.get("chunk_ids", []) for d in md.values()])):
                        graph_score[cid] += 0.5 * w

        nodes_out = sorted(
            [{"node": n, "category": self.G.nodes[n].get("category"),
              "score": round(s, 3), "n_chunks": len(node_prov.get(n, set()))}
             for n, s in node_scores.items()],
            key=lambda x: -x["score"])[:15]
        paths_out = [{"path": list(p), "weight": round(path_weight(p), 3)} for p in paths]
        return {
            "query_entities": q_ents,
            "nodes": nodes_out,
            "paths": paths_out,
            "chunk_ids": sorted(graph_score.keys()),
            "graph_score": {k: float(v) for k, v in graph_score.items()},
        }

graph_retriever = GraphRetriever(G, df_edges, df_entities, extract_entities_hybrid, chunk_meta)
gr = graph_retriever.retrieve("sudden severe headache and stiff neck")
print("Graph test 'sudden severe headache and stiff neck':")
print(" query entities:", gr["query_entities"])
print(" top nodes:", [(n['node'], n['score']) for n in gr["nodes"][:6]])
print(" top paths:", gr["paths"][:3])
print(" chunks touched by graph:", len(gr["chunk_ids"]))


Graph test 'sudden severe headache and stiff neck':
 query entities: ['headache']
 top nodes: [('ischemic stroke', 6.6), ('migraine', 6.0), ('multiple sclerosis', 4.8), ('alzheimer disease', 3.6), ('fever', 2.4), ('myasthenia gravis', 2.4)]
 top paths: []
 chunks touched by graph: 3884


### Cell 11 — Cross-Encoder + Graph-Aware Reranker

**Purpose:** Combine five signals into a final relevance score per candidate chunk:
- **dense** (hybrid dense component)
- **sparse** (BM25 component)
- **cross-encoder** (`cross-encoder/ms-marco-MiniLM-L-6-v2` if available, else a lexical-overlap proxy)
- **graph support** (from GraphRetriever)
- **evidence priority** (guideline > … > unknown)

Weights come from `CONFIG` (`w_dense`, `w_sparse`, `w_cross`, `w_graph`, `w_priority`).

**Inputs:** `hybrid_retriever`, `graph_retriever`, `embedder`.
**Outputs:** `reranker` with `.rerank(query, candidates, graph_evidence)`.
**Saved artifacts:** none.
**Customize later:** swap cross-encoder, learn weights on a labeled dev set.


In [13]:
# ===== Cell 11 - Cross-encoder + graph-aware reranker =====
class GraphAwareReranker:
    def __init__(self, embedder, weights=None):
        self.embedder = embedder
        w = weights or CONFIG
        self.w_dense = w["w_dense"]; self.w_sparse = w["w_sparse"]
        self.w_cross = w["w_cross"]; self.w_graph = w["w_graph"]
        self.w_priority = w["w_priority"]
        self.cross = None
        if HAVE_ST:
            try:
                self.cross = CrossEncoder(CONFIG["crossencoder_model_name"], max_length=512)
                print(f"[rerank] cross-encoder loaded: {CONFIG['crossencoder_model_name']}")
            except Exception as e:
                print(f"[rerank] cross-encoder load failed ({e}); using lexical-overlap proxy.")
        if self.cross is None:
            print("[rerank] using lexical-overlap cross-encoder proxy.")

    def _cross_scores(self, query, docs):
        if self.cross is not None:
            pairs = [(query, d) for d in docs]
            raw = self.cross.predict(pairs, show_progress_bar=False)
            raw = np.asarray(raw, dtype="float32")
            # sigmoid -> 0..1
            return 1.0 / (1.0 + np.exp(-raw))
        # proxy: token overlap F1
        qtok = set(tokenize(query))
        out = []
        for d in docs:
            dtok = set(tokenize(d))
            if not qtok or not dtok:
                out.append(0.0); continue
            inter = len(qtok & dtok)
            prec = inter / len(dtok); rec = inter / len(qtok)
            out.append(0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec))
        return np.asarray(out, dtype="float32")

    def rerank(self, query, candidates, graph_evidence):
        docs = [c["text"] for c in candidates]
        cross = self._cross_scores(query, docs)
        gmax = max(graph_evidence["graph_score"].values()) if graph_evidence["graph_score"] else 1e-9
        rows = []
        for i, c in enumerate(candidates):
            ds = c["dense_score"]; ss = c["sparse_score"]
            xs = float(cross[i])
            gs = float(graph_evidence["graph_score"].get(c["chunk_id"], 0.0)) / (gmax + 1e-9)
            pr = PRIORITY_WEIGHT.get(int(c["source_priority"]), 0.4)
            final = (self.w_dense * ds + self.w_sparse * ss + self.w_cross * xs +
                     self.w_graph * gs + self.w_priority * pr)
            c2 = dict(c)
            c2.update({"cross_score": xs, "graph_score": round(gs, 3),
                       "priority_score": pr, "final_score": round(final, 4)})
            rows.append(c2)
        rows.sort(key=lambda x: -x["final_score"])
        return rows[:CONFIG["top_k_rerank"]]

reranker = GraphAwareReranker(embedder)
# quick test
_cands = hybrid_retriever.search("stroke thrombolysis time window", k=10)
_gr = graph_retriever.retrieve("stroke thrombolysis time window")
_rer = reranker.rerank("stroke thrombolysis time window", _cands, _gr)
print("Rerank test 'stroke thrombolysis time window':")
for r in _rer:
    print(f"  final={r['final_score']:.3f} [d={r['dense_score']:.2f} s={r['sparse_score']:.2f} "
          f"x={r['cross_score']:.2f} g={r['graph_score']:.2f} p={r['priority_score']:.2f}] "
          f"[{r['chunk_id']}] {r['evidence_rank_label']}")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[rerank] cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Rerank test 'stroke thrombolysis time window':
  final=0.471 [d=0.51 s=1.00 x=0.42 g=0.03 p=0.40] [D223_c0] demo
  final=0.297 [d=0.47 s=0.00 x=0.38 g=0.03 p=0.40] [D051_c1] demo
  final=0.260 [d=0.46 s=0.59 x=0.04 g=0.01 p=0.40] [D143_c6] demo
  final=0.236 [d=0.49 s=0.46 x=0.00 g=0.03 p=0.40] [D051_c0] demo
  final=0.189 [d=0.56 s=0.00 x=0.01 g=0.03 p=0.40] [D051_c6] demo
  final=0.176 [d=0.51 s=0.00 x=0.00 g=0.05 p=0.40] [D051_c7] demo


### Cell 12 — Provenance-Preserving Evidence Aggregator

**Purpose:** Merge chunk evidence and graph evidence, de-duplicate, and **separate** the evidence
pack into *supporting*, *conflicting*, and *missing* buckets. Every retained item keeps its
provenance (`chunk_id`, `doc_id`, `source_priority`, evidence type).

**Inputs:** reranked candidates, `graph_evidence`, `df_entities`.
**Outputs:** `evidence_aggregator` producing an `EvidencePack` dict.
**Saved artifacts:** none (consumed by LLM + safety cells).
**Customize later:** conflict detection rules, contradiction model (NLI).


In [14]:
# ===== Cell 12 - Evidence aggregation =====
class EvidenceAggregator:
    def __init__(self, chunk_meta, df_entities):
        self.meta = chunk_meta
        self.ent_by_chunk = df_entities.groupby("chunk_id")["canonical"].apply(set).to_dict()

    def _conflict_signal(self, items):
        """Heuristic conflict detection: group by diagnosis entities and flag mutually exclusive
        treatments / red flags. Lightweight; replace with an NLI model later."""
        conflicts = []
        diag_groups = defaultdict(list)
        for it in items:
            ents = self.ent_by_chunk.get(it["chunk_id"], set())
            for e in ents:
                if ENTITY_LEXICON.get(e, {}).get("category") == "diagnosis":
                    diag_groups[e].append(it)
        # If two different high-priority sources disagree on a key fact we can't detect here;
        # we surface multi-diagnosis co-occurrence as potential differential tension.
        return conflicts

    def aggregate(self, reranked, graph_evidence, query_entities):
        seen = set()
        merged = []
        # add graph-only chunks not already in reranked (boost recall)
        existing = {r["chunk_id"] for r in reranked}
        for cid, gs in graph_evidence["graph_score"].items():
            if cid not in existing and cid in self.meta:
                m = self.meta[cid]
                merged.append({
                    "chunk_id": cid, "doc_id": m["doc_id"], "text": m["text"],
                    "title": m["title"], "source_priority": m["source_priority"],
                    "evidence_rank_label": m["evidence_rank_label"], "year": m["year"],
                    "final_score": round(0.3 * gs, 4), "from_graph": True,
                    "dense_score": 0.0, "sparse_score": 0.0,
                    "cross_score": 0.0, "graph_score": round(gs, 3),
                    "priority_score": PRIORITY_WEIGHT.get(int(m["source_priority"]), 0.4),
                })
        for r in reranked:
            r2 = dict(r); r2["from_graph"] = False
            merged.append(r2)
        # de-dup by chunk_id, keep best final_score
        merged.sort(key=lambda x: -x["final_score"])
        uniq = []
        for it in merged:
            if it["chunk_id"] in seen:
                continue
            seen.add(it["chunk_id"]); uniq.append(it)

        # supporting: top items with priority>=paper and graph/cross support
        supporting = [u for u in uniq if (u["priority_score"] >= 0.6 or u.get("graph_score", 0) > 0.1)
                      and u.get("final_score", 0) > 0.2][:8]
        # conflicting: heuristic placeholder
        conflicting = self._conflict_signal(uniq)
        # missing: query entities with no retrieved chunk coverage
        covered = set().union(*[self.ent_by_chunk.get(u["chunk_id"], set()) for u in uniq]) if uniq else set()
        missing = [e for e in query_entities if e not in covered]
        return {
            "supporting": supporting,
            "conflicting": conflicting,
            "missing_entities": missing,
            "all_ranked": uniq,
            "n_unique": len(uniq),
        }

evidence_aggregator = EvidenceAggregator(chunk_meta, df_entities)
_pack = evidence_aggregator.aggregate(_rer, _gr, graph_retriever._query_entities("stroke thrombolysis time window"))
print(f"Evidence pack: {len(_pack['supporting'])} supporting, {len(_pack['conflicting'])} conflicts, "
      f"{len(_pack['missing_entities'])} missing query entities")
for s in _pack["supporting"][:3]:
    print(f"  [supp] [{s['chunk_id']}] {s['evidence_rank_label']} :: {s['text'][:90]}...")


Evidence pack: 8 supporting, 0 conflicts, 0 missing query entities
  [supp] [D174_c1] demo :: 1. Core clinical responsibilities • Diagnose and manage acute and chronic neurologic disea...
  [supp] [D188_c0] demo :: user: What are some of the common conditions treated by neurologists? || assistant: Neurol...
  [supp] [D800_c1] demo :: Key associations include: ### **Stroke and Focal Brain Lesions** - **Right-hemisphere stro...


### Cell 13 — LLM Diagnostic Prompt Template (provider-agnostic + mock fallback)

**Purpose:** Build a strict, structured answer template and a provider-agnostic generator.
Default provider is `mock` (deterministic, offline) so the pipeline is fully runnable without keys.
Set `CONFIG["llm_provider"]` + the relevant `API_KEYS` to use a real LLM.

**Inputs:** `EvidencePack`, `CONFIG`, `API_KEYS`.
**Outputs:** `diagnostic_generator`.
**Saved artifacts:** none.
**Customize later:** add Gemini/Groq clients, prompt tuning, JSON-schema enforcement.


In [15]:
# ===== Cell 13 - LLM structured diagnostic template =====
DIAGNOSTIC_TEMPLATE = """You are a cautious neurology diagnostic assistant using ONLY the provided evidence.
Do NOT invent facts. If evidence is insufficient, say so explicitly.

PATIENT/QUERY:
{patient_block}

EVIDENCE (ranked, with provenance):
{evidence_block}

Respond STRICTLY in this template (fill each section; use 'Not enough evidence.' if needed):
1. Clinical summary: <2-3 sentences>
2. Most likely differentials: <comma-separated list, ranked>
3. Supporting evidence: <bullet list citing [CITEx]>
4. Contradictory or missing evidence: <bullet list or 'None identified.'>
5. Recommended next diagnostic steps: <bullet list>
6. Red flags / urgent escalation: <bullet list or 'None identified.'>
7. Confidence: <one of: low | medium | high> - <one-line justification>
8. Citations: <list of [CITEx] used>
9. Disclaimer: <fixed text: not a substitute for clinician judgment>
"""

class DiagnosticGenerator:
    def __init__(self, provider=CONFIG["llm_provider"], model=CONFIG["llm_model"]):
        self.provider = provider
        # Pick a sensible model per provider when the configured one does not match.
        if provider == "gemini" and not str(model).startswith("gemini"):
            model = "gemini-1.5-flash"
        elif provider == "groq" and str(model).startswith("gemini"):
            model = "llama-3.3-70b-versatile"
        self.model = model
        self._client = None
        if provider == "groq" and API_KEYS.get("GROQ_API_KEY"):
            try:
                from groq import Groq
                self._client = Groq(api_key=API_KEYS["GROQ_API_KEY"])
                print(f"[llm] groq client ready (model={self.model}).")
            except Exception as e:
                print(f"[llm] groq client init failed ({e}); using mock.")
                self.provider = "mock"
        elif provider == "gemini" and API_KEYS.get("GEMINI_API_KEY"):
            try:
                import google.generativeai as genai
                genai.configure(api_key=API_KEYS["GEMINI_API_KEY"])
                self._client = genai.GenerativeModel(self.model)
                print(f"[llm] gemini client ready (model={self.model}).")
            except Exception as e:
                print(f"[llm] gemini client init failed ({e}); using mock.")
                self.provider = "mock"
        else:
            print(f"[llm] provider='{provider}' (no key) -> using deterministic MOCK generator.")
            self.provider = "mock"

    def _build_prompt(self, query, patient_context, pack):
        pblock = f"Query: {query}\n"
        if patient_context:
            pblock += "Patient context: " + "; ".join(f"{k}={v}" for k, v in patient_context.items())
        eblock_lines = []
        for i, e in enumerate(pack["supporting"], 1):
            eblock_lines.append(
                f"[CITE{i}] ({e['evidence_rank_label']}, priority={e['source_priority']}) "
                f"{e['title']} :: {e['text']}")
        eblock = "\n".join(eblock_lines) or "No supporting evidence retrieved."
        return DIAGNOSTIC_TEMPLATE.format(patient_block=pblock, evidence_block=eblock)

    def _mock_generate(self, query, patient_context, pack):
        """Deterministic offline generator: templates the 9 sections from the evidence pack."""
        supp = pack["supporting"]
        cite = lambda i: f"[CITE{i}]"
        summary = f"Query concerns: {query}. "
        if supp:
            tier = supp[0].get('evidence_rank_label') or supp[0].get('evidence_tier') or 'unknown'
            summary += f"Top evidence ({tier}) indicates: {supp[0]['text'][:160]}"
        # When the corpus is synthetic/dialog, make the demo framing explicit
        tiers = {(s.get('evidence_rank_label') or s.get('evidence_tier') or '') for s in supp}
        is_demo_corpus = any('demo' in str(t) or 'synthetic' in str(t) for t in tiers)
        diffs = sorted({e["title"] for e in supp})[:5] or ["Insufficient evidence for a differential."]
        sup_lines = [f"- {cite(i)} {e['evidence_rank_label']}: {e['text'][:140]}" for i, e in enumerate(supp, 1)] or ["- None retrieved."]
        contra = []
        if pack["missing_entities"]:
            contra.append("- Missing coverage for entities: " + ", ".join(pack["missing_entities"]))
        contra = contra or ["- None identified."]
        steps = ["- Confirm key clinical signs and time of onset (if applicable).",
                 "- Obtain relevant neuroimaging / labs as supported by evidence.",
                 "- Correlate with exam findings and specialist consultation."]
        redflags = []
        for e in supp:
            for ent in evidence_aggregator.ent_by_chunk.get(e["chunk_id"], set()):
                if ENTITY_LEXICON.get(ent, {}).get("category") == "red_flag":
                    redflags.append(f"- {ent} (per {e['chunk_id']})")
        redflags = redflags or ["- None identified."]
        conf = "medium" if len(supp) >= 3 else ("low" if supp else "low")
        cites = [cite(i) for i in range(1, len(supp) + 1)] or ["[none]"]
        text = textwrap.dedent(f"""\
1. Clinical summary: {summary}
2. Most likely differentials: {', '.join(diffs)}
3. Supporting evidence:
{chr(10).join(sup_lines)}
4. Contradictory or missing evidence:
{chr(10).join(contra)}
5. Recommended next diagnostic steps:
{chr(10).join(steps)}
6. Red flags / urgent escalation:
{chr(10).join(redflags)}
7. Confidence: {conf} - based on {len(supp)} supporting item(s) and evidence priority.
8. Citations: {', '.join(cites)}
9. Disclaimer: This is a research prototype and NOT a substitute for clinician judgment.
{' Evidence is drawn from a synthetic/demo neurology corpus, not clinical guidelines.' if is_demo_corpus else ''}
""")
        return {"answer": text, "citations": cites, "confidence": conf, "provider": "mock"}

    def generate(self, query, patient_context, pack):
        prompt = self._build_prompt(query, patient_context, pack)
        if self.provider == "mock":
            out = self._mock_generate(query, patient_context, pack)
            out["prompt"] = prompt
            return out
        # --- Real LLM paths (only used when key configured) ---
        try:
            if self.provider == "groq":
                resp = self._client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=CONFIG["max_tokens"], temperature=0.2)
                ans = resp.choices[0].message.content
            elif self.provider == "gemini":
                resp = self._client.generate_content(
                    prompt,
                    generation_config={"max_output_tokens": CONFIG["max_tokens"],
                                       "temperature": 0.2})
                ans = resp.text
            else:
                raise RuntimeError("unsupported provider")
            return {"answer": ans, "prompt": prompt, "provider": self.provider,
                    "confidence": "unknown", "citations": []}
        except Exception as e:
            print(f"[llm] real provider failed ({e}); falling back to mock.")
            return self.generate(query, patient_context, pack) if self.provider != "mock" else self._mock_generate(query, patient_context, pack)

diagnostic_generator = DiagnosticGenerator()
print(f"Diagnostic generator ready: provider={diagnostic_generator.provider}")


[llm] provider='mock' (no key) -> using deterministic MOCK generator.
Diagnostic generator ready: provider=mock


### Cell 14 — Safety Verifier

**Purpose:** Post-generation safety checks on the answer + evidence: unsupported claims,
missing citations, overconfidence, contradiction, missing urgent escalation, unsafe treatment language.
Includes **neurology emergency rules** (stroke, status epilepticus, meningitis, raised ICP,
thunderclap headache) that force an *escalation* flag.

**Inputs:** generator output, `EvidencePack`.
**Outputs:** `safety_verifier` returning a `SafetyReport`.
**Saved artifacts:** none.
**Customize later:** add more emergency rules, an NLI-based contradiction detector.


In [16]:
# ===== Cell 14 - Safety verifier =====
NEURO_EMERGENCY_RULES = [
    {"id": "stroke", "triggers": ["stroke", "thrombolysis", "thrombectomy", "last known well"],
     "must_mention": ["time", "4.5", "thrombolysis", "ct"],
     "escalation": "Suspected acute stroke: confirm time of last known well, non-contrast CT, and thrombolysis/thrombectomy eligibility IMMEDIATELY."},
    {"id": "status_epilepticus", "triggers": ["status epilepticus", "seizure lasting", "continuous seizure"],
     "must_mention": ["benzodiazepine", "lorazepam", "diazepam", "midazolam"],
     "escalation": "Suspected status epilepticus: administer IV benzodiazepine per protocol and escalate to ICU if refractory."},
    {"id": "meningitis", "triggers": ["meningitis", "neck stiffness", "petechial"],
     "must_mention": ["ceftriaxone", "vancomycin", "dexamethasone", "antibiotic"],
     "escalation": "Suspected bacterial meningitis: start empiric antibiotics (ceftriaxone + vancomycin +/- dexamethasone) WITHOUT delay."},
    {"id": "raised_icp", "triggers": ["raised intracranial pressure", "raised icp", "midline shift", "cushing triad", "papilledema"],
     "must_mention": ["mannitol", "hypertonic saline", "neurosurgery", "ct"],
     "escalation": "Suspected raised intracranial pressure: urgent CT, osmotic therapy, and neurosurgical consult."},
    {"id": "thunderclap_headache", "triggers": ["thunderclap", "sudden severe headache", "worst headache"],
     "must_mention": ["ct", "subarachnoid", "lumbar puncture", "xanthochromia"],
     "escalation": "Suspected subarachnoid hemorrhage: emergency non-contrast CT; if negative, lumbar puncture for xanthochromia."},
]

UNSAFE_TREATMENT_PATTERNS = [
    r"\bdefinitely (cure|treat)\b", r"\bguaranteed\b", r"\bno need (for|to) (see|consult)\b",
    r"\bignore\b.{0,20}(emergency|doctor|physician)", r"\bstop taking\b",
]

class SafetyVerifier:
    def verify(self, query, gen_out, pack):
        ans = gen_out.get("answer", "").lower()
        cites = gen_out.get("citations", [])
        conf = gen_out.get("confidence", "unknown")
        issues = []
        escalations = []

        # 1) unsupported claim / missing citation: any 'CITE' number used without evidence list
        used_cite_ids = set(re.findall(r"\[cite(\d+)\]", ans))
        n_supp = len(pack["supporting"])
        if used_cite_ids and any(int(n) > n_supp for n in used_cite_ids):
            issues.append({"type": "unsupported_claim", "detail": "Citation index exceeds evidence count."})
        if pack["supporting"] and not used_cite_ids and "[none]" not in cites:
            issues.append({"type": "missing_citations", "detail": "Answer has supporting evidence but no citations."})

        # 2) overconfidence
        if conf == "high" and n_supp < 3:
            issues.append({"type": "overconfidence", "detail": "High confidence with <3 supporting items."})
        if re.search(r"\bdefinitely\b|\bcertainly\b|\bguaranteed\b", ans):
            issues.append({"type": "overconfident_language", "detail": "Absolutist language detected."})

        # 3) contradiction with evidence (very light heuristic)
        if pack["conflicting"]:
            issues.append({"type": "evidence_conflict", "detail": str(pack["conflicting"])})

        # 4) missing urgent escalation / neurology emergencies
        qtext = (query + " " + ans).lower()
        for rule in NEURO_EMERGENCY_RULES:
            if any(t in qtext for t in rule["triggers"]):
                if not any(m in ans for m in rule["must_mention"]):
                    escalations.append(rule["escalation"])
                    issues.append({"type": "missing_escalation", "rule": rule["id"],
                                   "detail": f"Trigger matched but no required mention of {rule['must_mention']}."})

        # 5) unsafe treatment language
        for pat in UNSAFE_TREATMENT_PATTERNS:
            if re.search(pat, ans):
                issues.append({"type": "unsafe_treatment_language", "detail": pat})

        passed = (len(issues) == 0)
        return {"passed": passed, "issues": issues, "escalations": escalations,
                "n_issues": len(issues), "n_escalations": len(escalations)}

safety_verifier = SafetyVerifier()
# quick test on a mock generation
_tgen = diagnostic_generator.generate("stroke thrombolysis", None, _pack)
_trep = safety_verifier.verify("stroke thrombolysis", _tgen, _pack)
print("Safety test 'stroke thrombolysis':", "PASS" if _trep["passed"] else f"{_trep['n_issues']} issue(s)")
for iss in _trep["issues"][:5]:
    print("   -", iss["type"], "::", iss.get("detail", "")[:80])
for esc in _trep["escalations"]:
    print("   ESCALATION:", esc)


Safety test 'stroke thrombolysis': PASS


### Cell 15 — Final Inference Pipeline

**Purpose:** Wire every component into one entry point `run_pipeline(query, patient_context=None)`
that returns a structured result: ranked evidence, evidence pack, generated answer, citations,
confidence, and safety report. This is the single function the rest of the notebook (and a future
serving layer) calls.

**Inputs:** all upstream components.
**Outputs:** `run_pipeline(...)`.
**Saved artifacts:** none (demo + eval call it).
**Customize later:** caching, async batching, logging, telemetry.


In [17]:
# ===== Cell 15 - Final inference pipeline =====
def run_pipeline(query, patient_context=None,
                 k_hybrid=CONFIG["top_k_hybrid"], k_rerank=CONFIG["top_k_rerank"],
                 verbose=True):
    t0 = time.time()
    # 1) hybrid retrieval
    candidates = hybrid_retriever.search(query, k=k_hybrid)
    # 2) graph / path retrieval
    graph_evidence = graph_retriever.retrieve(query)
    query_entities = graph_evidence["query_entities"]
    # 3) rerank (cross-encoder + graph-aware)
    reranked = reranker.rerank(query, candidates, graph_evidence)
    # 4) evidence aggregation (provenance-preserving)
    pack = evidence_aggregator.aggregate(reranked, graph_evidence, query_entities)
    # 5) LLM structured diagnostic
    gen = diagnostic_generator.generate(query, patient_context, pack)
    # 6) safety verification
    report = safety_verifier.verify(query, gen, pack)
    dt = time.time() - t0
    result = {
        "query": query,
        "patient_context": patient_context or {},
        "query_entities": query_entities,
        "graph_nodes": graph_evidence["nodes"],
        "graph_paths": graph_evidence["paths"],
        "reranked": reranked,
        "evidence_pack": pack,
        "answer": gen["answer"],
        "citations": gen["citations"],
        "confidence": gen["confidence"],
        "safety": report,
        "provider": gen.get("provider", CONFIG["llm_provider"]),
        "elapsed_sec": round(dt, 3),
    }
    if verbose:
        print(f"[pipeline] {dt:.2f}s | ents={query_entities} | supp={len(pack['supporting'])} "
              f"| issues={report['n_issues']} | escalations={report['n_escalations']} | provider={result['provider']}")
    return result

print("run_pipeline() ready.")


run_pipeline() ready.


### Cell 16 — Demo Queries / Case Vignettes

**Purpose:** Run a few realistic neurology queries/vignettes through the full pipeline and pretty-print
the final answer, citations, top evidence, and any safety warnings.

**Inputs:** `run_pipeline`.
**Outputs:** printed vignette results + `demo_results` list.
**Saved artifacts:** `artifacts/demo_results.json`.
**Customize later:** more vignettes, golden-answer comparison.


In [18]:
# ===== Cell 16 - Demo queries / case vignettes =====
DEMOS = [
    {"query": "A 68-year-old has sudden right-sided weakness and slurred speech that started 90 minutes ago. What is the likely diagnosis and immediate management?",
     "patient_context": {"age": 68, "onset_min": 90, "symptoms": "right weakness, slurred speech"}},
    {"query": "Patient with thunderclap headache reaching maximum intensity in 30 seconds, now with neck stiffness and photophobia. Differential and workup?",
     "patient_context": {"onset": "thunderclap", "features": "neck stiffness, photophobia"}},
    {"query": "Seizure activity has continued for 10 minutes despite a benzodiazepine. Next steps?",
     "patient_context": {"duration_min": 10, "first_line": "benzodiazepine given"}},
    {"query": "Young adult with fever, confusion and neck stiffness. What is the empiric treatment?",
     "patient_context": {"fever": True, "features": "confusion, neck stiffness"}},
    {"query": "Summarize the neurology conversations in the corpus about headache with photophobia and nausea.",
     "patient_context": {"corpus_note": "Demo query retrieving synthetic conversation evidence"}},
]

demo_results = []
for d in DEMOS:
    print("\n" + "=" * 78)
    print("QUERY:", d["query"])
    if d.get("patient_context"):
        print("CONTEXT:", d["patient_context"])
    print("-" * 78)
    res = run_pipeline(d["query"], d.get("patient_context"))
    demo_results.append(res)
    print("\n--- ANSWER ---")
    print(res["answer"])
    print("\n--- CITATIONS ---", res["citations"])
    print("\n--- TOP EVIDENCE ---")
    for e in res["reranked"][:3]:
        print(f"  [{e['chunk_id']}] {e['evidence_rank_label']} :: {e['text'][:100]}...")
    if res["safety"]["escalations"]:
        print("\n--- SAFETY ESCALATIONS ---")
        for esc in res["safety"]["escalations"]:
            print("  !!", esc)
    if res["safety"]["issues"]:
        print("\n--- SAFETY ISSUES ---")
        for iss in res["safety"]["issues"]:
            print("  -", iss["type"], "::", iss.get("detail", "")[:80])

with open(ART_DIR / "demo_results.json", "w") as f:
    # strip large fields for readability
    slim = [{k: v for k, v in r.items() if k in {"query", "query_entities", "answer", "citations",
              "confidence", "safety", "provider", "elapsed_sec"}} for r in demo_results]
    json.dump(slim, f, indent=2)
print("\nSaved demo results ->", ART_DIR / "demo_results.json")



QUERY: A 68-year-old has sudden right-sided weakness and slurred speech that started 90 minutes ago. What is the likely diagnosis and immediate management?
CONTEXT: {'age': 68, 'onset_min': 90, 'symptoms': 'right weakness, slurred speech'}
------------------------------------------------------------------------------
[pipeline] 0.69s | ents=['flaccid paralysis'] | supp=8 | issues=0 | escalations=0 | provider=mock

--- ANSWER ---
1. Clinical summary: Query concerns: A 68-year-old has sudden right-sided weakness and slurred speech that started 90 minutes ago. What is the likely diagnosis and immediate management?. Top evidence (demo) indicates: user: What are some of the common conditions treated by neurologists? || assistant: Neurologists specialize in disorders of the brain, spinal cord, nerves, and 
2. Most likely differentials: How can the pattern of muscle weakness provide diagnostic cl..., How does exercise impact the weakness associated with myasth..., What are some of the common

### Cell 17 — Evaluation Block

**Purpose:** Notebook-friendly retrieval + answer metrics. Uses mock golden labels (chunk-id sets per
query) since the dataset may be synthetic. Reports **Recall@k, Precision@k, MRR**, plus answer-side
proxies: **faithfulness proxy, unsupported-claim rate, citation coverage, answer completeness**.

**Inputs:** `run_pipeline`, `df_chunks`, a small golden set.
**Outputs:** `eval_report` dict + printed table.
**Saved artifacts:** `artifacts/evaluation_report.json`.
**Customize later:** replace golden set with expert labels; add RAGAS-style metrics.


In [19]:
# ===== Cell 17 - Evaluation block =====
# Golden (query -> relevant chunk_ids). Derived AUTOMATICALLY from the actual df_chunks so it works
# for the HF conversational neurology dataset (where doc_ids/titles
# are derived from each conversation's first user message).
# Replace with expert-annotated labels for a real neurology evidence corpus.
def _auto_golden(df_chunks, df_clean, queries):
    """For each query, find chunks whose doc title/source contains informative keywords.
    Falls back to evenly-spaced slice when no keyword match."""
    gold = {}
    n_docs = df_chunks["doc_id"].nunique()
    for q in queries:
        q_words = set(re.findall(r"[a-z]+", q.lower()))
        matched = set()
        for _, r in df_chunks.iterrows():
            title = str(r.get("title", "")).lower()
            text = str(r.get("text", "")).lower()
            if len(q_words & (set(title.split()) | set(text.split()))) >= 2:
                matched.add(r["chunk_id"])
        if matched:
            gold[q] = list(matched)[:4]
        else:
            idxs = list(range(0, n_docs, max(1, n_docs // 4)))[:4]
            gold[q] = [df_chunks[df_chunks["doc_id"] == f"D{i:03d}"]["chunk_id"].iloc[0]
                       for i in idxs if not df_chunks[df_chunks["doc_id"] == f"D{i:03d}"].empty]
    return gold

_GOLDEN_QUERIES = [
    "sudden right-sided weakness and slurred speech acute stroke management",
    "thunderclap headache subarachnoid hemorrhage workup",
    "status epilepticus refractory management second line",
    "fever neck stiffness confusion empiric antibiotics meningitis",
    "headache seizure weakness neurology differential",
]
GOLDEN = _auto_golden(df_chunks, df_clean, _GOLDEN_QUERIES)

def evaluate(k=5):
    rows = []
    recall_k, precision_k, mrr = [], [], []
    faith_proxy, unsupported_rate, citation_cov, completeness = [], [], [], []
    for q, gold in GOLDEN.items():
        gold = set(gold)
        res = run_pipeline(q, None, verbose=False)
        retrieved = [r["chunk_id"] for r in res["reranked"]][:k]
        rel = [1 if c in gold else 0 for c in retrieved]
        rec = len(gold & set(retrieved)) / max(1, len(gold))
        prec = sum(rel) / max(1, len(retrieved))
        rr = next((1.0 / (i + 1) for i, x in enumerate(rel) if x), 0.0)
        recall_k.append(rec); precision_k.append(prec); mrr.append(rr)
        # answer-side proxies
        ans = res["answer"].lower()
        # faithfulness proxy: fraction of citations whose chunk text shares tokens with answer
        cites_in_ans = set(re.findall(r"\[cite(\d+)\]", ans))
        supp = res["evidence_pack"]["supporting"]
        if supp and cites_in_ans:
            shared = 0
            for n in cites_in_ans:
                idx = int(n) - 1
                if 0 <= idx < len(supp):
                    toks = set(tokenize(supp[idx]["text"])) & set(tokenize(ans))
                    if toks:
                        shared += 1
            faith = shared / max(1, len(cites_in_ans))
        else:
            faith = 0.0
        faith_proxy.append(faith)
        unsupported_rate.append(1.0 if res["safety"]["n_issues"] > 0 else 0.0)
        citation_cov.append(len(cites_in_ans) / max(1, len(supp))) if supp else citation_cov.append(0.0)
        # completeness: count of non-empty template sections 1..6
        secs = re.findall(r"^\s*\d+\.\s", res["answer"], flags=re.MULTILINE)
        completeness.append(len(secs) / 6.0)
        rows.append({"query": q[:50], "recall@k": round(rec, 2), "precision@k": round(prec, 2),
                     "mrr": round(rr, 2), "faithfulness": round(faith, 2),
                     "completeness": round(len(secs) / 6.0, 2)})

    df_eval = pd.DataFrame(rows)
    summary = {
        "k": k, "n_queries": len(rows),
        "mean_recall@k": round(float(np.mean(recall_k)), 3),
        "mean_precision@k": round(float(np.mean(precision_k)), 3),
        "mean_mrr": round(float(np.mean(mrr)), 3),
        "mean_faithfulness_proxy": round(float(np.mean(faith_proxy)), 3),
        "unsupported_claim_rate": round(float(np.mean(unsupported_rate)), 3),
        "mean_citation_coverage": round(float(np.mean(citation_cov)), 3),
        "mean_answer_completeness": round(float(np.mean(completeness)), 3),
    }
    return df_eval, summary

eval_df, eval_report = evaluate(k=5)
print("\n--- Per-query metrics ---")
print(eval_df.to_string(index=False))
print("\n--- Aggregate ---")
for k, v in eval_report.items():
    print(f"  {k}: {v}")
with open(ART_DIR / "evaluation_report.json", "w") as f:
    json.dump({"summary": eval_report, "per_query": eval_df.to_dict("records")}, f, indent=2)
print("\nSaved evaluation ->", ART_DIR / "evaluation_report.json")



--- Per-query metrics ---
                                             query  recall@k  precision@k  mrr  faithfulness  completeness
sudden right-sided weakness and slurred speech acu      0.00          0.0  0.0           1.0           1.5
thunderclap headache subarachnoid hemorrhage worku      0.75          0.6  1.0           1.0           1.5
status epilepticus refractory management second li      0.00          0.0  0.0           1.0           1.5
fever neck stiffness confusion empiric antibiotics      0.00          0.0  0.0           1.0           1.5
  headache seizure weakness neurology differential      0.00          0.0  0.0           1.0           1.5

--- Aggregate ---
  k: 5
  n_queries: 5
  mean_recall@k: 0.15
  mean_precision@k: 0.12
  mean_mrr: 0.2
  mean_faithfulness_proxy: 1.0
  unsupported_claim_rate: 0.2
  mean_citation_coverage: 1.0
  mean_answer_completeness: 1.5

Saved evaluation -> /kaggle/working/artifacts/evaluation_report.json


### Cell 18 — Save Artifacts

**Purpose:** Persist every major intermediate artifact under `/kaggle/working/artifacts/` so the
pipeline is fully reproducible and inspectable. (Most cells already save incrementally; this cell
writes a consolidated manifest and any remaining artifacts.)

**Inputs:** all upstream objects.
**Outputs:** `artifacts/manifest.json` + zipped bundle.
**Saved artifacts:** consolidated manifest + `neural_graphrag_artifacts.zip`.
**Customize later:** push to a Kaggle Dataset, S3, or model registry.


In [20]:
# ===== Cell 18 - Save artifacts =====
import shutil

# Re-save graph + embeddings + manifest for a clean consolidated bundle
with open(ART_DIR / "kg.gpickle", "wb") as f:
    pickle.dump(G, f)
np.save(ART_DIR / "chunk_embeddings.npy", chunk_embeddings)

manifest = {
    "created_on": pd.Timestamp.now().isoformat(),
    "is_kaggle": bool(IS_KAGGLE),
    "config": CONFIG,
    "components": {
        "corpus_docs": int(len(df_clean)),
        "chunks": int(len(df_chunks)),
        "entities_mentions": int(len(df_entities)),
        "unique_entities": int(df_entities["canonical"].nunique()),
        "kg_nodes": int(G.number_of_nodes()),
        "kg_edges": int(G.number_of_edges()),
        "embed_dim": int(chunk_embeddings.shape[1]),
        "faiss_backend": "faiss" if HAVE_FAISS else "numpy",
        "sparse_backend": sparse_retriever.backend,
        "embedder_backend": embedder.backend,
        "llm_provider": diagnostic_generator.provider,
        "cross_encoder": reranker.cross is not None,
    },
    "evaluation_summary": eval_report,
    "files": sorted([str(p.relative_to(WORK_DIR)) for p in ART_DIR.rglob("*") if p.is_file()]),
}
with open(ART_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Zip everything for easy download from the Kaggle UI
zip_path = WORK_DIR / "neural_graphrag_artifacts"
shutil.make_archive(str(zip_path), "zip", ART_DIR)

print("=== Artifact manifest ===")
print(json.dumps(manifest, indent=2)[:1500], "...")
print("\nArtifacts directory:", ART_DIR)
print("Consolidated bundle:", str(zip_path) + ".zip")
for p in sorted(ART_DIR.glob("*")):
    print(f"  {p.name:30s} {p.stat().st_size:>10,} bytes")
print("\nAll artifacts saved. Pipeline complete.")


=== Artifact manifest ===
{
  "created_on": "2026-07-11T12:55:17.696635",
  "is_kaggle": true,
  "config": {
    "chunk_size": 320,
    "chunk_overlap": 60,
    "top_k_hybrid": 12,
    "top_k_rerank": 6,
    "embed_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "crossencoder_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "embed_dim": 384,
    "dense_weight": 0.6,
    "sparse_weight": 0.4,
    "w_dense": 0.25,
    "w_sparse": 0.15,
    "w_cross": 0.35,
    "w_graph": 0.15,
    "w_priority": 0.1,
    "llm_provider": "mock",
    "llm_model": "llama-3.3-70b-versatile",
    "max_tokens": 700
  },
  "components": {
    "corpus_docs": 1428,
    "chunks": 12846,
    "entities_mentions": 5643,
    "unique_entities": 86,
    "kg_nodes": 86,
    "kg_edges": 940,
    "embed_dim": 384,
    "faiss_backend": "faiss",
    "sparse_backend": "bm25",
    "embedder_backend": "st",
    "llm_provider": "mock",
    "cross_encoder": true
  },
  "evaluation_summary": {
    "k": 5,
    "n_

## Wrap-up

### 1) ASCII architecture diagram

```
                    +--------------------------+
   Neurology Corpus | cleaning + dedup +       |
        ------>     | evidence-type ranking    |  Cell 1,2
                    +-----------+--------------+
                                |
                                v
                    +--------------------------+
                    | semantic chunking        |  Cell 3
                    +-----------+--------------+
                                |
                                v
                    +--------------------------+
                    | entity normalization     |  Cell 4
                    +-----+--------------+-----+
                          |              |
                          v              v
              +-------------------+  +----------------------+
   embeddings |  Cell 6           |  | neurology KG (nx)    | Cell 5
   + FAISS    |  Cell 7           |  |                      |
              +---------+---------+  +----------+-----------+
                        |                       |
   BM25/sparse Cell 8   |                       |
                        v                       |
              +-------------------+             |
              | hybrid retriever  |<------------+  (graph_evidence joins at rerank)
              | Cell 9            |             |
              +---------+---------+             |
                        |                       |
                        v                       v
              +-----------------------------------+
              | graph/path retriever (Cell 10)    |
              +----------------+------------------+
                               |
                               v
              +-----------------------------------+
              | cross-encoder + graph-aware       | Cell 11
              | reranker                          |
              +----------------+------------------+
                               |
                               v
              +-----------------------------------+
              | evidence aggregator              | Cell 12
              | (supporting / conflicting /      |
              |  missing, provenance preserved)   |
              +----------------+------------------+
                               |
                               v
              +-----------------------------------+
              | LLM structured diagnostic (Cell 13)
              +----------------+------------------+
                               |
                               v
              +-----------------------------------+
              | safety verifier + neuro emergencies| Cell 14
              +----------------+------------------+
                               |
                               v
              +-----------------------------------+
              | run_pipeline() final answer       | Cell 15
              | (citations / confidence / escal.) |
              +-----------------------------------+
```

### 2) Execution order summary

1. **Cell 0** – install + imports + `CONFIG`/`API_KEYS` + paths.
2. **Cell 1** – parse HF `messages` into corpus (HF-only, no fallback) → `df_corpus`.
3. **Cell 2** – clean + dedup (hash + fuzzy) + evidence priority → `df_clean`.
4. **Cell 3** – semantic chunking → `df_chunks`.
5. **Cell 4** – entity normalization (lexicon + optional scispaCy) → `df_entities`.
6. **Cell 5** – KG build (typed relations + provenance) → `G`, `df_edges`, `df_nodes`.
7. **Cell 6** – pluggable embeddings → `embedder`, `chunk_embeddings`.
8. **Cell 7** – FAISS / numpy-cosine index → `faiss_index`.
9. **Cell 8** – BM25 / sparse retriever → `sparse_retriever`.
10. **Cell 9** – hybrid fusion → `hybrid_retriever`.
11. **Cell 10** – graph/path retrieval → `graph_retriever`.
12. **Cell 11** – cross-encoder + graph-aware reranker → `reranker`.
13. **Cell 12** – provenance-preserving aggregator → `evidence_aggregator`.
14. **Cell 13** – structured diagnostic generator (provider-agnostic + mock) → `diagnostic_generator`.
15. **Cell 14** – safety verifier → `safety_verifier`.
16. **Cell 15** – `run_pipeline(query, patient_context)` end-to-end.
17. **Cell 16** – demo vignettes.
18. **Cell 17** – evaluation (Recall/Precision/MRR + answer proxies).
19. **Cell 18** – save artifacts + manifest + zip.

### 3) Where to plug in API keys later

Set these as **Kaggle Secrets** (Add-ons → Secrets) and read them in **Cell 0**, or via env vars:

| Key | Used by | How to enable |
|-----|---------|---------------|
| `GROQ_API_KEY` | `DiagnosticGenerator` (Cell 13) | `LLM_PROVIDER="groq"` (default); model `llama-3.3-70b-versatile` |
| `GEMINI_API_KEY` | `DiagnosticGenerator` (Cell 13) | `LLM_PROVIDER="gemini"`; model `gemini-1.5-flash` |
| `HF_TOKEN` | HF embedding/cross-encoder models (Cell 6/11) | set `CONFIG["embed_model_name"]` / `crossencoder_model_name` to gated HF models |
| `PINECONE_API_KEY` | optional external vector DB | TODO: replace `faiss_index` with Pinecone in Cell 7 |

> OpenAI / Anthropic are **not** used as default providers. Set `LLM_PROVIDER` (env or Kaggle
> Secret) to `groq` or `gemini` and supply the matching key; otherwise the pipeline runs with
> a deterministic mock generator.

```python
# In Cell 0 (Kaggle), add a Secret named LLM_PROVIDER + the matching key:
# from kagglesecrets import UserSecretsClient
# sc = UserSecretsClient()
# API_KEYS["GROQ_API_KEY"] = sc.get_secret("GROQ_API_KEY")   # or GEMINI_API_KEY
# import os; os.environ["LLM_PROVIDER"] = "groq"             # or "gemini"
# LLM_PROVIDER = "groq"; CONFIG["llm_provider"] = LLM_PROVIDER
```

### 4) How to convert this notebook into a modular Python package later

1. Create package `neural_graphrag/` with submodules mirroring cells:
   - `data.py` (Cells 1–2), `chunking.py` (3), `entities.py` (4), `kg.py` (5),
     `embed.py` (6), `indexes.py` (7–8), `retrieval.py` (9–10), `rerank.py` (11),
     `evidence.py` (12), `generation.py` (13), `safety.py` (14), `pipeline.py` (15).
2. Move `CONFIG`, `API_KEYS`, `ENTITY_LEXICON`, `PRIORITY_MAP`, `NEURO_EMERGENCY_RULES`
   into `config.py`.
3. Replace notebook globals with a `Pipeline` dataclass holding `embedder`, `faiss_index`,
   `sparse_retriever`, `G`, etc.; `Pipeline.run(query, patient_context)` mirrors `run_pipeline`.
4. Add `pyproject.toml`, `tests/` (port the golden set from Cell 17), and a thin CLI
   (`python -m neural_graphrag query "..."`).
5. Keep the LLM provider behind an interface (`LLMClient.generate`) so Groq/Gemini/mock
   are interchangeable; add a `serve.py` (FastAPI) only when a frontend is needed.

> **Optional extension hook (similar patient retrieval):** the paper's similar-patient module maps
> cleanly onto a future `patients.py` that embeds structured patient vectors and reuses the
> cross-encoder reranker. It is intentionally left as a `TODO` to keep this build pipeline-first.
